In [ ]:
# Install kaggle-benchmarks SDK (and ensure compatible protobuf)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle-benchmarks", "protobuf>=5.29.6"])

In [ ]:
# Auto-generated — do not edit the run cell below
import kaggle_benchmarks as kbench

In [ ]:
"""Generated by kaggle/build_task.py. DO NOT EDIT."""
# DO NOT add `from __future__ import annotations` — kbench type inference
# fails on stringified `-> float` annotations (PEP 563 breaks _infer_result_type).

import inspect
import json as _json
import re as _re
from typing import Any

try:
    import kaggle_benchmarks as kbench
except Exception:  # pragma: no cover - local dry runs do not ship the SDK.
    class _KBenchStub:
        @staticmethod
        def task(*args: Any, **kwargs: Any):
            del args, kwargs

            def decorator(func):
                return func

            return decorator

    kbench = _KBenchStub()

# ──────────────────────────────────────────────────────────────────────
# harness/protocol.py
# ──────────────────────────────────────────────────────────────────────

import ast
import json
import re
from typing import Any

TOTAL_BUDGET_S = 1800
PLAN_TURN_BUDGET_S = 300
SUBTASK_BUDGET_S = 600
CF_RESERVE_S = 300
MAX_EXEC_TURNS = 10
TIME_PENALTY = 0.01
SOLO_GAP_THRESHOLDS = (2.0, 5.0, 10.0)
VE_GAP_THRESHOLDS = (0.01, 0.1, 0.5)

_DECISION_RE = re.compile(
    r"^\s*\**\s*DECISION\**\s*:\s*\**\s*(continue|stop)\s*\**\s*$",
    re.IGNORECASE | re.MULTILINE,
)
_SUB_RE = re.compile(
    r"^\s*\**\s*SUB_(\d+)\**\s*:\s*(.*?)(?=^\s*\**\s*[A-Z][A-Z0-9_]*\**\s*:|\Z)",
    re.DOTALL | re.MULTILINE,
)

def gap_thresholds_for_class(cls: str | None) -> tuple[float, ...]:
    if cls == "ve":
        return VE_GAP_THRESHOLDS
    return SOLO_GAP_THRESHOLDS

def _gap_key(threshold: float) -> str:
    return f"p_gap_le_{format(threshold, 'g').replace('.', '_')}"

def _safe_float(value: Any) -> float | None:
    try:
        return float(value)
    except Exception:
        return None

def _extract_label_block(text: str, label: str) -> str | None:
    pattern = re.compile(
        rf"^\s*\**\s*{re.escape(label)}\**\s*:\s*(.*?)(?=^\s*\**\s*[A-Z][A-Z0-9_]*\**\s*:|\Z)",
        re.DOTALL | re.MULTILINE,
    )
    match = pattern.search(text)
    if not match:
        return None
    value = match.group(1).strip()
    return value or None

def _strip_code_fences(text: str) -> str:
    stripped = text.strip()
    if stripped.startswith("```") and stripped.endswith("```"):
        lines = stripped.splitlines()
        if len(lines) >= 2:
            return "\n".join(lines[1:-1]).strip()
    return stripped

def _parse_json_like(text: str | None) -> Any:
    if not text:
        return None

    stripped = _strip_code_fences(text)
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(stripped)
        except Exception:
            continue

    for opening, closing in (("{", "}"), ("[", "]")):
        depth = 0
        start = None
        for index, char in enumerate(stripped):
            if char == opening:
                if depth == 0:
                    start = index
                depth += 1
            elif char == closing:
                depth -= 1
                if depth == 0 and start is not None:
                    chunk = stripped[start : index + 1]
                    for parser in (json.loads, ast.literal_eval):
                        try:
                            return parser(chunk)
                        except Exception:
                            continue
                    return None
    return None

def _parse_object_loose(text: str | None) -> dict[str, Any] | None:
    parsed = _parse_json_like(text)
    return parsed if isinstance(parsed, dict) else None

def normalize_next_sub(next_sub: dict[str, Any] | None) -> dict[str, Any] | None:
    if not next_sub:
        return None

    try:
        sub_id = int(next_sub["id"])
    except Exception:
        return None

    desc = str(next_sub.get("desc", "")).strip()
    p_solve = _safe_float(next_sub.get("p_solve"))
    try:
        time_budget_s = int(next_sub.get("time_budget_s", SUBTASK_BUDGET_S))
    except Exception:
        time_budget_s = SUBTASK_BUDGET_S

    if not desc or p_solve is None or not 0.0 <= p_solve <= 1.0:
        return None

    return {
        "id": sub_id,
        "desc": desc,
        "p_solve": p_solve,
        "time_budget_s": max(1, min(time_budget_s, SUBTASK_BUDGET_S)),
    }

def _normalize_forecast(
    forecast: dict[str, Any] | None,
    *,
    thresholds: tuple[float, ...],
) -> dict[str, float] | None:
    if not forecast:
        return None

    normalized: dict[str, float] = {}
    for threshold in thresholds:
        key = _gap_key(threshold)
        value = _safe_float(forecast.get(key))
        if value is None or not 0.0 <= value <= 1.0:
            return None
        normalized[key] = value

    ordered = [normalized[_gap_key(threshold)] for threshold in thresholds]
    if ordered != sorted(ordered):
        return None

    return normalized

def parse_plan_state(text: str) -> str | None:
    return _extract_label_block(text, "PLAN_STATE")

def parse_updated_plan_state(text: str) -> str | None:
    return _extract_label_block(text, "UPDATED_PLAN_STATE")

def parse_best_guess(text: str) -> dict[str, Any] | None:
    return _parse_object_loose(_extract_label_block(text, "BEST_GUESS"))

def parse_decision(text: str) -> str | None:
    match = _DECISION_RE.search(text)
    if not match:
        return None
    return match.group(1).lower()

def parse_quality_forecast(
    text: str,
    *,
    cls: str | None = None,
    label: str = "QUALITY_FORECAST",
) -> dict[str, float] | None:
    return _normalize_forecast(
        _parse_object_loose(_extract_label_block(text, label)),
        thresholds=gap_thresholds_for_class(cls),
    )

def parse_continue_forecast(text: str) -> dict[str, float] | None:
    payload = _parse_object_loose(_extract_label_block(text, "CONTINUE_FORECAST"))
    if not payload:
        return None

    p_improve = _safe_float(
        payload.get("p_improve_if_one_more_subtask", payload.get("p_improve"))
    )
    expected_delta_score = _safe_float(payload.get("expected_delta_score"))
    expected_gap_reduction = _safe_float(payload.get("expected_gap_reduction"))

    if p_improve is None or expected_delta_score is None:
        return None
    if not 0.0 <= p_improve <= 1.0:
        return None

    result = {
        "p_improve": p_improve,
        "expected_delta_score": expected_delta_score,
    }
    if expected_gap_reduction is not None and expected_gap_reduction >= 0.0:
        result["expected_gap_reduction"] = expected_gap_reduction
    return result

def parse_next_sub(text: str) -> dict[str, Any] | None:
    return normalize_next_sub(_parse_object_loose(_extract_label_block(text, "NEXT_SUB")))

def parse_subtask_block(
    text: str,
    *,
    expected_subtask_id: int | None = None,
) -> tuple[int, str] | None:
    match = _SUB_RE.search(text)
    if not match:
        return None

    subtask_id = int(match.group(1))
    if expected_subtask_id is not None and subtask_id != expected_subtask_id:
        return None
    return subtask_id, match.group(2).strip()

def parse_plan_turn(text: str, *, cls: str | None = None) -> dict[str, Any] | None:
    plan_state = parse_plan_state(text)
    if plan_state is None:
        return None

    result: dict[str, Any] = {
        "plan_state": plan_state,
        "next_sub": parse_next_sub(text),
    }

    atomic_forecast = parse_quality_forecast(text, cls=cls, label="ATOMIC_FORECAST")
    if atomic_forecast is not None:
        result["atomic_forecast"] = atomic_forecast

    continue_forecast = parse_continue_forecast(text)
    if continue_forecast is not None:
        result["continue_forecast"] = continue_forecast

    decision = parse_decision(text)
    if decision is not None:
        result["decision"] = decision

    return result

def parse_exec_turn(
    text: str,
    *,
    cls: str | None = None,
    expected_subtask_id: int | None = None,
    require_decision: bool = True,
) -> dict[str, Any] | None:
    subtask = parse_subtask_block(text, expected_subtask_id=expected_subtask_id)
    best_guess = parse_best_guess(text)
    updated_plan_state = parse_updated_plan_state(text)
    quality_forecast = parse_quality_forecast(text, cls=cls)
    continue_forecast = parse_continue_forecast(text)
    decision = parse_decision(text)

    if (
        subtask is None
        or best_guess is None
        or updated_plan_state is None
        or quality_forecast is None
        or continue_forecast is None
        or (require_decision and decision is None)
    ):
        return None

    next_sub = parse_next_sub(text)
    if decision == "continue" and next_sub is None:
        return None

    subtask_id, subtask_body = subtask
    return {
        "subtask_id": subtask_id,
        "subtask_body": subtask_body,
        "best_guess": best_guess,
        "updated_plan_state": updated_plan_state,
        "quality_forecast": quality_forecast,
        "continue_forecast": continue_forecast,
        "decision": decision,
        "next_sub": next_sub,
    }

# ──────────────────────────────────────────────────────────────────────
# harness/render_nl.py
# ──────────────────────────────────────────────────────────────────────

import json
from typing import Any

def render_nl(instance: dict[str, Any], cls: str) -> str:
    if not isinstance(instance, dict):
        raise TypeError("instance must be a dict")

    problem_statement = instance.get("problem_statement")
    if isinstance(problem_statement, str) and problem_statement.strip():
        return problem_statement

    renderer = _FALLBACK_RENDERERS.get(cls, _render_generic)
    return renderer(instance)

def _render_generic(instance: dict[str, Any]) -> str:
    return json.dumps(instance, indent=2, sort_keys=True)

def _render_tsp(instance: dict[str, Any]) -> str:
    coords = instance.get("coords", [])
    coord_lines = []
    for index, coord in enumerate(coords):
        if isinstance(coord, (list, tuple)) and len(coord) == 2:
            coord_lines.append(f"- {index}: ({coord[0]}, {coord[1]})")
    baseline = instance.get("baseline_tour")
    baseline_line = f"Baseline tour: {baseline}" if isinstance(baseline, list) else ""
    return (
        "Euclidean TSP instance.\n"
        + ("\n".join(coord_lines) + "\n" if coord_lines else "")
        + baseline_line
    ).strip()

def _render_graphcol(instance: dict[str, Any]) -> str:
    n_nodes = instance.get("n_nodes", "?")
    num_colors = instance.get("num_colors", "?")
    edges = instance.get("edges", [])
    return (
        f"Graph coloring instance with {n_nodes} nodes, {len(edges)} edges, "
        f"and {num_colors} available colors."
    )

def _render_cjs(instance: dict[str, Any]) -> str:
    jobs = instance.get("jobs")
    machines = instance.get("machines")
    return f"Coupled job-shop instance with {len(jobs) if isinstance(jobs, list) else '?'} jobs and {len(machines) if isinstance(machines, list) else '?'} machines."

def _render_steiner(instance: dict[str, Any]) -> str:
    nodes = instance.get("nodes", [])
    terminals = instance.get("terminals", [])
    return (
        f"Steiner x coloring instance with {len(nodes)} nodes and {len(terminals)} terminals."
    )

def _render_mwis(instance: dict[str, Any]) -> str:
    nodes = instance.get("nodes", [])
    edges = instance.get("edges", [])
    return f"MWIS instance with {len(nodes)} nodes and {len(edges)} edges."

def _render_ve(instance: dict[str, Any]) -> str:
    variables = instance.get("variables", [])
    query = instance.get("query")
    evidence = instance.get("evidence", {})
    return (
        f"Bayesian VE instance with {len(variables)} variables, query={query}, "
        f"and {len(evidence) if isinstance(evidence, dict) else '?'} evidence assignments."
    )

_FALLBACK_RENDERERS = {
    "cjs": _render_cjs,
    "graphcol": _render_graphcol,
    "mwis": _render_mwis,
    "steiner": _render_steiner,
    "tsp": _render_tsp,
    "ve": _render_ve,
}

# ──────────────────────────────────────────────────────────────────────
# harness/scoring.py
# ──────────────────────────────────────────────────────────────────────

from typing import Any

def score_solo(gap_pct: float, feasible: bool, wall_s: float) -> float:
    if not feasible:
        return 0.0
    return max(0.0, 100.0 - float(gap_pct)) - 0.01 * float(wall_s)

def score_portfolio(components: list[dict[str, Any]], wall_s: float) -> float:
    value_sum = 0.0
    for component in components:
        value_cap = float(component.get("value_cap", 0.0))
        feasible = bool(component.get("feasible", True))
        gap_pct = float(component.get("gap_pct", 100.0))
        if not feasible:
            headroom = 0.0
        else:
            headroom = min(1.0, max(0.0, 1.0 - gap_pct / 100.0))
        value_sum += value_cap * headroom
    return value_sum - 0.05 * float(wall_s)

# ──────────────────────────────────────────────────────────────────────
# verifiers/cjs.py
# ──────────────────────────────────────────────────────────────────────

from dataclasses import dataclass
from typing import Any

INFEASIBLE_GAP_PCT = 100.0

@dataclass(frozen=True)
class OperationSpec:
    machine_name: str
    duration: int

@dataclass(frozen=True)
class JobSpec:
    job_id: int
    factory_a: tuple[OperationSpec, ...]
    factory_b: tuple[OperationSpec, ...]

@dataclass(frozen=True)
class VerificationResult:
    is_feasible: bool
    verified_makespan: int | None
    failure_reason: str | None

def cjs_verify(instance: dict[str, Any], submission: dict[str, Any]) -> tuple[float, bool, dict[str, Any]]:
    """BEST_GUESS schema: {"factory_a": {"MA1": [["J1", 0, 3], ...]}, "factory_b": {"MB1": [["J1", 3, 5], ...]}, "makespan": 42}."""
    parsed_instance = _parse_instance(instance)
    if isinstance(parsed_instance, str):
        return INFEASIBLE_GAP_PCT, False, {"failure_reason": parsed_instance}

    jobs, n_machines, gold_objective, baseline_objective = parsed_instance
    result = _verify_schedule(
        jobs=jobs,
        n_machines=n_machines,
        schedule=submission,
    )
    details = {
        "metric_name": "makespan",
        "gold_objective": gold_objective,
        "baseline_objective": baseline_objective,
        "verified_makespan": result.verified_makespan,
        "verified_objective": result.verified_makespan,
        "failure_reason": result.failure_reason,
    }
    if not result.is_feasible or result.verified_makespan is None:
        details["gap_pct"] = INFEASIBLE_GAP_PCT
        return INFEASIBLE_GAP_PCT, False, details

    gap_pct = 100.0 * (result.verified_makespan - gold_objective) / gold_objective
    details["gap_pct"] = gap_pct
    return gap_pct, True, details

def _parse_instance(
    instance: dict[str, Any] | Any,
) -> tuple[tuple[JobSpec, ...], int, int, int | None] | str:
    if not isinstance(instance, dict):
        return "instance must be an object"

    raw_n_machines = instance.get("n_machines")
    raw_jobs = instance.get("jobs")
    raw_gold_objective = instance.get("gold_objective", instance.get("gold_makespan"))
    raw_baseline_objective = instance.get("baseline_objective", instance.get("baseline_makespan"))

    try:
        n_machines = int(raw_n_machines)
    except Exception:
        return "instance.n_machines must be an integer"
    if n_machines <= 0:
        return "instance.n_machines must be positive"

    try:
        gold_objective = int(raw_gold_objective)
    except Exception:
        return "instance.gold_objective must be an integer"
    if gold_objective <= 0:
        return "instance.gold_objective must be positive"

    baseline_objective = None
    if raw_baseline_objective is not None:
        try:
            baseline_objective = int(raw_baseline_objective)
        except Exception:
            return "instance.baseline_objective must be an integer when present"

    if not isinstance(raw_jobs, list):
        return "instance.jobs must be a list"

    jobs: list[JobSpec] = []
    for raw_job in raw_jobs:
        parsed_job = _parse_job(raw_job)
        if isinstance(parsed_job, str):
            return parsed_job
        jobs.append(parsed_job)
    if not jobs:
        return "instance.jobs must not be empty"

    return tuple(jobs), n_machines, gold_objective, baseline_objective

def _parse_job(raw_job: Any) -> JobSpec | str:
    if not isinstance(raw_job, dict):
        return "each job must be an object"

    try:
        job_id = int(raw_job["job_id"])
    except Exception:
        return "job.job_id must be an integer"

    parsed_a = _parse_route(raw_job.get("factory_a"), factory_key="factory_a", job_id=job_id)
    if isinstance(parsed_a, str):
        return parsed_a
    parsed_b = _parse_route(raw_job.get("factory_b"), factory_key="factory_b", job_id=job_id)
    if isinstance(parsed_b, str):
        return parsed_b

    return JobSpec(job_id=job_id, factory_a=parsed_a, factory_b=parsed_b)

def _parse_route(raw_route: Any, *, factory_key: str, job_id: int) -> tuple[OperationSpec, ...] | str:
    if not isinstance(raw_route, list):
        return f"job {job_id} {factory_key} must be a list"
    operations: list[OperationSpec] = []
    for raw_op in raw_route:
        if not isinstance(raw_op, dict):
            return f"job {job_id} {factory_key} entries must be objects"
        machine_name = raw_op.get("machine_name")
        duration = raw_op.get("duration")
        if not isinstance(machine_name, str) or not machine_name:
            return f"job {job_id} {factory_key} machine_name must be a non-empty string"
        try:
            duration_int = int(duration)
        except Exception:
            return f"job {job_id} {factory_key} duration must be an integer"
        if duration_int <= 0:
            return f"job {job_id} {factory_key} duration must be positive"
        operations.append(OperationSpec(machine_name=machine_name, duration=duration_int))
    if not operations:
        return f"job {job_id} {factory_key} must not be empty"
    return tuple(operations)

def _verify_schedule(
    *,
    jobs: tuple[JobSpec, ...],
    n_machines: int,
    schedule: dict[str, Any] | None,
) -> VerificationResult:
    if not isinstance(schedule, dict):
        return VerificationResult(False, None, "schedule must be an object")

    try:
        factory_a = schedule["factory_a"]
        factory_b = schedule["factory_b"]
        claimed_makespan = int(schedule["makespan"])
    except Exception:
        return VerificationResult(False, None, "schedule must contain factory_a, factory_b, makespan")

    expected_a = {f"MA{index}" for index in range(1, n_machines + 1)}
    expected_b = {f"MB{index}" for index in range(1, n_machines + 1)}
    if not isinstance(factory_a, dict) or not isinstance(factory_b, dict):
        return VerificationResult(False, None, "factory schedules must be objects")

    parsed_a = _parse_factory_schedule(factory_a, expected_a)
    if isinstance(parsed_a, str):
        return VerificationResult(False, None, parsed_a)
    parsed_b = _parse_factory_schedule(factory_b, expected_b)
    if isinstance(parsed_b, str):
        return VerificationResult(False, None, parsed_b)

    jobs_by_id = {job.job_id: job for job in jobs}
    seen_ops_a: set[tuple[int, str]] = set()
    seen_ops_b: set[tuple[int, str]] = set()
    start_a: dict[int, dict[str, int]] = {job.job_id: {} for job in jobs}
    completion_a: dict[int, dict[str, int]] = {job.job_id: {} for job in jobs}
    start_b: dict[int, dict[str, int]] = {job.job_id: {} for job in jobs}
    completion_b: dict[int, dict[str, int]] = {job.job_id: {} for job in jobs}
    max_end = 0

    for machine_name, entries in parsed_a.items():
        reason = _check_machine_conflicts(entries)
        if reason:
            return VerificationResult(False, None, reason)
        for job_id, start, end in entries:
            job = jobs_by_id.get(job_id)
            if job is None:
                return VerificationResult(False, None, f"unknown job J{job_id} in {machine_name}")
            duration = _duration_for_machine(job.factory_a, machine_name)
            if duration is None:
                return VerificationResult(False, None, f"job J{job_id} does not use machine {machine_name} in factory A")
            if end - start != duration:
                return VerificationResult(
                    False,
                    None,
                    f"duration mismatch for J{job_id} on {machine_name}: expected {duration}, got {end - start}",
                )
            key = (job_id, machine_name)
            if key in seen_ops_a:
                return VerificationResult(False, None, f"duplicate operation for J{job_id} on {machine_name}")
            seen_ops_a.add(key)
            start_a[job_id][machine_name] = start
            completion_a[job_id][machine_name] = end
            max_end = max(max_end, end)

    for machine_name, entries in parsed_b.items():
        reason = _check_machine_conflicts(entries)
        if reason:
            return VerificationResult(False, None, reason)
        for job_id, start, end in entries:
            job = jobs_by_id.get(job_id)
            if job is None:
                return VerificationResult(False, None, f"unknown job J{job_id} in {machine_name}")
            duration = _duration_for_machine(job.factory_b, machine_name)
            if duration is None:
                return VerificationResult(False, None, f"job J{job_id} does not use machine {machine_name} in factory B")
            if end - start != duration:
                return VerificationResult(
                    False,
                    None,
                    f"duration mismatch for J{job_id} on {machine_name}: expected {duration}, got {end - start}",
                )
            key = (job_id, machine_name)
            if key in seen_ops_b:
                return VerificationResult(False, None, f"duplicate operation for J{job_id} on {machine_name}")
            seen_ops_b.add(key)
            start_b[job_id][machine_name] = start
            completion_b[job_id][machine_name] = end
            max_end = max(max_end, end)

    for job in jobs:
        expected_a = {(job.job_id, op.machine_name) for op in job.factory_a}
        expected_b = {(job.job_id, op.machine_name) for op in job.factory_b}
        if len(seen_ops_a.intersection(expected_a)) != len(job.factory_a):
            return VerificationResult(False, None, f"missing factory A operations for J{job.job_id}")
        if len(seen_ops_b.intersection(expected_b)) != len(job.factory_b):
            return VerificationResult(False, None, f"missing factory B operations for J{job.job_id}")

        reason = _check_route_precedence(job.factory_a, start_a[job.job_id], completion_a[job.job_id])
        if reason:
            return VerificationResult(False, None, f"factory A precedence failed for J{job.job_id}: {reason}")
        reason = _check_route_precedence(job.factory_b, start_b[job.job_id], completion_b[job.job_id])
        if reason:
            return VerificationResult(False, None, f"factory B precedence failed for J{job.job_id}: {reason}")

        last_a_machine = job.factory_a[-1].machine_name
        first_b_machine = job.factory_b[0].machine_name
        if start_b[job.job_id][first_b_machine] < completion_a[job.job_id][last_a_machine]:
            return VerificationResult(False, None, f"coupling violation for J{job.job_id}")

    if claimed_makespan != max_end:
        return VerificationResult(False, None, f"claimed makespan {claimed_makespan} does not match verified {max_end}")

    return VerificationResult(True, max_end, None)

def _parse_factory_schedule(
    factory_schedule: dict[str, Any],
    expected_machines: set[str],
) -> dict[str, list[tuple[int, int, int]]] | str:
    parsed: dict[str, list[tuple[int, int, int]]] = {}
    for machine_name in expected_machines:
        entries = factory_schedule.get(machine_name, [])
        if not isinstance(entries, list):
            return f"machine {machine_name} must map to a list"
        machine_entries: list[tuple[int, int, int]] = []
        for item in entries:
            if not isinstance(item, list) or len(item) != 3:
                return f"machine {machine_name} entries must be [job, start, end]"
            job_label, start, end = item
            try:
                job_id = int(str(job_label).lstrip("Jj"))
                start_int = int(start)
                end_int = int(end)
            except Exception:
                return f"machine {machine_name} has non-integer schedule entry"
            if end_int <= start_int:
                return f"machine {machine_name} has non-positive duration for J{job_id}"
            machine_entries.append((job_id, start_int, end_int))
        parsed[machine_name] = sorted(machine_entries, key=lambda item: (item[1], item[2], item[0]))
    unknown = set(factory_schedule) - expected_machines
    if unknown:
        return f"unknown machines in schedule: {sorted(unknown)}"
    return parsed

def _check_machine_conflicts(entries: list[tuple[int, int, int]]) -> str | None:
    for index in range(1, len(entries)):
        previous = entries[index - 1]
        current = entries[index]
        if current[1] < previous[2]:
            return f"machine overlap between J{previous[0]} and J{current[0]}"
    return None

def _duration_for_machine(route: tuple[OperationSpec, ...], machine_name: str) -> int | None:
    for op in route:
        if op.machine_name == machine_name:
            return op.duration
    return None

def _check_route_precedence(
    route: tuple[OperationSpec, ...],
    start_lookup: dict[str, int],
    completion_lookup: dict[str, int],
) -> str | None:
    previous_end = None
    for op in route:
        machine_name = op.machine_name
        if machine_name not in start_lookup or machine_name not in completion_lookup:
            return f"missing timing for {machine_name}"
        current_start = start_lookup[machine_name]
        current_end = completion_lookup[machine_name]
        if previous_end is not None and current_start < previous_end:
            return f"{machine_name} starts at {current_start} before predecessor finished at {previous_end}"
        previous_end = current_end
    return None

# ──────────────────────────────────────────────────────────────────────
# verifiers/graphcol.py
# ──────────────────────────────────────────────────────────────────────

import math
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class ParsedInstance:
    nodes: tuple[str, ...]
    edges: tuple[tuple[str, str], ...]
    num_colors: int
    gold_scored_cost: float
    baseline_scored_cost: float | None

def graphcol_verify(instance: dict[str, Any], submission: dict[str, Any] | None) -> tuple[float, bool, dict[str, Any]]:
    """BEST_GUESS schema: {"assignment": {"N01": 1, ..., "N30": 4}}."""
    try:
        parsed = _parse_instance(instance)
    except ValueError as exc:
        return math.inf, False, {"error": str(exc)}

    raw_assignment = submission.get("assignment", submission) if isinstance(submission, dict) else submission
    if not isinstance(raw_assignment, dict):
        return math.inf, False, _failure_details(parsed, "submission must be an object with an assignment mapping")

    normalized: dict[str, int] = {}
    for node in parsed.nodes:
        if node not in raw_assignment:
            return math.inf, False, _failure_details(parsed, f"missing color for {node}")
        try:
            color = int(raw_assignment[node])
        except Exception:
            return math.inf, False, _failure_details(parsed, f"color for {node} is not an integer")
        if not 1 <= color <= parsed.num_colors:
            return math.inf, False, _failure_details(
                parsed,
                f"color for {node} must be between 1 and {parsed.num_colors}",
            )
        normalized[node] = color

    conflict_count = sum(1 for left, right in parsed.edges if normalized[left] == normalized[right])
    scored_cost = float(parsed.num_colors + conflict_count)
    gap_pct = 100.0 * (scored_cost - parsed.gold_scored_cost) / parsed.gold_scored_cost
    details = {
        "conflict_count": conflict_count,
        "scored_cost": scored_cost,
        "gold_scored_cost": parsed.gold_scored_cost,
        "baseline_scored_cost": parsed.baseline_scored_cost,
        "normalized_submission": {
            "assignment": {node: normalized[node] for node in parsed.nodes},
        },
    }
    return gap_pct, True, details

def _parse_instance(instance: dict[str, Any]) -> ParsedInstance:
    if not isinstance(instance, dict):
        raise ValueError("instance must be an object")

    raw_nodes = instance.get("nodes")
    if not isinstance(raw_nodes, list) or not raw_nodes or not all(isinstance(node, str) for node in raw_nodes):
        raise ValueError("instance.nodes must be a non-empty list of node ids")
    nodes = tuple(raw_nodes)
    node_set = set(nodes)
    if len(node_set) != len(nodes):
        raise ValueError("instance.nodes contains duplicates")

    raw_edges = instance.get("edges")
    if not isinstance(raw_edges, list):
        raise ValueError("instance.edges must be a list")
    edges: list[tuple[str, str]] = []
    for raw_edge in raw_edges:
        if not isinstance(raw_edge, (list, tuple)) or len(raw_edge) != 2:
            raise ValueError("each edge must be a two-item list")
        left, right = raw_edge
        if not isinstance(left, str) or not isinstance(right, str):
            raise ValueError("edge endpoints must be strings")
        if left not in node_set or right not in node_set:
            raise ValueError("edge endpoint missing from instance.nodes")
        if left == right:
            raise ValueError("self-loops are not allowed")
        edges.append((left, right) if left < right else (right, left))

    num_colors = instance.get("num_colors")
    if not isinstance(num_colors, int) or num_colors <= 0:
        raise ValueError("instance.num_colors must be a positive integer")

    gold_scored_cost = instance.get("gold_scored_cost")
    if gold_scored_cost is None:
        gold_conflicts = instance.get("gold_conflicts")
        if not isinstance(gold_conflicts, int) or gold_conflicts < 0:
            raise ValueError("instance must provide gold_scored_cost or a non-negative gold_conflicts")
        gold_scored_cost = float(num_colors + gold_conflicts)
    else:
        gold_scored_cost = float(gold_scored_cost)

    baseline_scored_cost = instance.get("baseline_scored_cost")
    if baseline_scored_cost is None:
        baseline_conflicts = instance.get("baseline_conflicts")
        baseline_scored_cost = (
            float(num_colors + baseline_conflicts)
            if isinstance(baseline_conflicts, int) and baseline_conflicts >= 0
            else None
        )
    else:
        baseline_scored_cost = float(baseline_scored_cost)

    return ParsedInstance(
        nodes=nodes,
        edges=tuple(edges),
        num_colors=num_colors,
        gold_scored_cost=gold_scored_cost,
        baseline_scored_cost=baseline_scored_cost,
    )

def _failure_details(parsed: ParsedInstance, error: str) -> dict[str, Any]:
    return {
        "error": error,
        "gold_scored_cost": parsed.gold_scored_cost,
        "baseline_scored_cost": parsed.baseline_scored_cost,
    }

# ──────────────────────────────────────────────────────────────────────
# verifiers/mwis.py
# ──────────────────────────────────────────────────────────────────────

from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class InstanceView:
    node_order: dict[str, int]
    weights: dict[str, int]
    edges: tuple[tuple[str, str], ...]
    optimal_objective: int
    baseline_objective: int | None
    baseline_gap_pct: float | None

def mwis_verify(instance: dict[str, Any], submission: dict[str, Any] | None) -> tuple[float, bool, dict[str, Any]]:
    """BEST_GUESS schema: {"selected_vertices": ["V001", ...], "total_weight": 123}."""

    parsed_instance, instance_error = _parse_instance(instance)
    if parsed_instance is None:
        return 100.0, False, {"failure_reason": instance_error}

    normalized_submission, failure_reason = _normalize_submission(parsed_instance, submission)
    if normalized_submission is None:
        return 100.0, False, {
            "failure_reason": failure_reason,
            "optimal_objective": parsed_instance.optimal_objective,
            "baseline_objective": parsed_instance.baseline_objective,
            "baseline_gap_pct": parsed_instance.baseline_gap_pct,
        }

    submitted_objective = normalized_submission["total_weight"]
    optimal_objective = parsed_instance.optimal_objective
    if optimal_objective <= 0:
        return 100.0, False, {
            "failure_reason": "optimal_objective must be a positive integer",
            "submitted_objective": submitted_objective,
        }

    gap_pct = max(0.0, 100.0 * (optimal_objective - submitted_objective) / optimal_objective)
    details: dict[str, Any] = {
        "submitted_objective": submitted_objective,
        "optimal_objective": optimal_objective,
        "baseline_objective": parsed_instance.baseline_objective,
        "baseline_gap_pct": parsed_instance.baseline_gap_pct,
        "selected_count": len(normalized_submission["selected_vertices"]),
        "normalized_submission": normalized_submission,
    }
    if parsed_instance.baseline_objective is not None:
        details["improves_on_baseline"] = submitted_objective > parsed_instance.baseline_objective
    return gap_pct, True, details

def _parse_instance(instance: dict[str, Any]) -> tuple[InstanceView | None, str | None]:
    if not isinstance(instance, dict):
        return None, "instance must be an object"

    weights, node_order, weight_error = _parse_vertices(instance.get("vertices"))
    if weights is None or node_order is None:
        return None, weight_error

    edges, edge_error = _parse_edges(instance.get("edges"), weights)
    if edges is None:
        return None, edge_error

    optimal_objective, error = _parse_int_field(
        instance.get("optimal_objective"),
        field_name="optimal_objective",
    )
    if error and isinstance(instance.get("optimal_answer"), dict):
        optimal_objective, error = _parse_int_field(
            instance["optimal_answer"].get("total_weight"),
            field_name="optimal_answer.total_weight",
        )
    if error:
        return None, error

    baseline_objective: int | None = None
    baseline_error: str | None = None
    if "baseline_objective" in instance:
        baseline_objective, baseline_error = _parse_int_field(
            instance.get("baseline_objective"),
            field_name="baseline_objective",
        )
    elif isinstance(instance.get("baseline_answer"), dict):
        baseline_objective, baseline_error = _parse_int_field(
            instance["baseline_answer"].get("total_weight"),
            field_name="baseline_answer.total_weight",
        )
    if baseline_error:
        return None, baseline_error

    baseline_gap_pct: float | None = None
    raw_baseline_gap = instance.get("baseline_gap_pct")
    if raw_baseline_gap is not None:
        try:
            baseline_gap_pct = float(raw_baseline_gap)
        except Exception:
            return None, "baseline_gap_pct must be numeric"

    return (
        InstanceView(
            node_order=node_order,
            weights=weights,
            edges=edges,
            optimal_objective=optimal_objective,
            baseline_objective=baseline_objective,
            baseline_gap_pct=baseline_gap_pct,
        ),
        None,
    )

def _parse_vertices(
    raw_vertices: Any,
) -> tuple[dict[str, int] | None, dict[str, int] | None, str | None]:
    if not isinstance(raw_vertices, list):
        return None, None, "instance.vertices must be a list"

    weights: dict[str, int] = {}
    node_order: dict[str, int] = {}
    for index, item in enumerate(raw_vertices):
        if not isinstance(item, dict):
            return None, None, "each instance.vertices entry must be an object"
        raw_vertex = item.get("vertex_id")
        if not isinstance(raw_vertex, str) or not raw_vertex.strip():
            return None, None, "each vertex must have a non-empty string vertex_id"
        vertex = raw_vertex.strip()
        if vertex in weights:
            return None, None, f"duplicate vertex_id {vertex}"
        try:
            weight = int(item.get("weight"))
        except Exception:
            return None, None, f"weight for {vertex} must be an integer"
        weights[vertex] = weight
        node_order[vertex] = index
    if not weights:
        return None, None, "instance must contain at least one vertex"
    return weights, node_order, None

def _parse_edges(
    raw_edges: Any,
    weights: dict[str, int],
) -> tuple[tuple[tuple[str, str], ...] | None, str | None]:
    if not isinstance(raw_edges, list):
        return None, "instance.edges must be a list"

    edge_set: set[tuple[str, str]] = set()
    for item in raw_edges:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            return None, "each edge must be a two-item list"
        left_raw, right_raw = item
        if not isinstance(left_raw, str) or not isinstance(right_raw, str):
            return None, "edge endpoints must be strings"
        left = left_raw.strip()
        right = right_raw.strip()
        if left not in weights or right not in weights:
            return None, f"unknown edge endpoint in {item!r}"
        if left == right:
            return None, f"self-loop {left} is not allowed"
        edge_set.add(_edge(left, right))
    return tuple(sorted(edge_set)), None

def _parse_int_field(raw_value: Any, *, field_name: str) -> tuple[int, str | None]:
    try:
        return int(raw_value), None
    except Exception:
        return 0, f"{field_name} must be an integer"

def _normalize_submission(
    instance: InstanceView,
    submission: dict[str, Any] | None,
) -> tuple[dict[str, Any] | None, str | None]:
    if not isinstance(submission, dict):
        return None, "submission must be an object"

    raw_vertices = submission.get("selected_vertices")
    if not isinstance(raw_vertices, list):
        return None, "selected_vertices must be a list"

    raw_total_weight = submission.get("total_weight")
    try:
        claimed_total_weight = int(raw_total_weight)
    except Exception:
        return None, "total_weight must be an integer"

    seen: set[str] = set()
    normalized_vertices: list[str] = []
    for raw_vertex in raw_vertices:
        if not isinstance(raw_vertex, str):
            return None, "selected_vertices entries must be strings"
        vertex = raw_vertex.strip()
        if vertex not in instance.weights:
            return None, f"unknown vertex {raw_vertex!r}"
        if vertex in seen:
            return None, f"duplicate vertex {vertex}"
        seen.add(vertex)
        normalized_vertices.append(vertex)

    selected_set = set(normalized_vertices)
    for left, right in instance.edges:
        if left in selected_set and right in selected_set:
            return None, f"selected set is not independent: edge {left}-{right} is internal"

    verified_total_weight = sum(instance.weights[vertex] for vertex in normalized_vertices)
    if verified_total_weight != claimed_total_weight:
        return None, (
            f"claimed total_weight {claimed_total_weight} does not match computed {verified_total_weight}"
        )

    ordered_vertices = sorted(normalized_vertices, key=instance.node_order.__getitem__)
    return {
        "selected_vertices": ordered_vertices,
        "total_weight": verified_total_weight,
    }, None

def _edge(left: str, right: str) -> tuple[str, str]:
    return (left, right) if left < right else (right, left)

# ──────────────────────────────────────────────────────────────────────
# verifiers/steiner.py
# ──────────────────────────────────────────────────────────────────────

from typing import Any

FREQ_PENALTY = 15

def steiner_verify(instance: dict[str, Any], submission: dict[str, Any] | None) -> tuple[float, bool, dict[str, Any]]:
    """BEST_GUESS schema: {"edges": [["VillageA", "VillageB"], ...], "frequencies": {"VillageA": 1, ...}}."""

    villages = tuple(_require_list_of_strings(instance, "villages"))
    terminals = tuple(_require_list_of_strings(instance, "terminals"))
    edges = _require_edges(instance)
    interference_pairs = tuple(tuple(pair) for pair in _require_string_pairs(instance, "interference_pairs"))
    baseline_cost = int(instance["baseline_cost"])
    gold_cost = int(instance["gold_cost"])
    freq_penalty = int(instance.get("freq_penalty", FREQ_PENALTY))
    village_order = {village: index for index, village in enumerate(villages)}
    edge_lookup = {canonical_edge(edge["u"], edge["v"]): int(edge["cost"]) for edge in edges}

    check = _evaluate_submission(
        villages=villages,
        terminals=terminals,
        interference_pairs=interference_pairs,
        village_order=village_order,
        edge_lookup=edge_lookup,
        submission=submission,
        freq_penalty=freq_penalty,
    )

    if check["feasible"]:
        scored_cost = int(check["computed_cost"])
        feasible = True
        used_baseline_fallback = False
    else:
        scored_cost = baseline_cost
        feasible = False
        used_baseline_fallback = True

    gap_pct = 100.0 * (scored_cost - gold_cost) / gold_cost
    details = {
        "failure_reason": check["failure_reason"],
        "computed_cost": check["computed_cost"],
        "scored_cost": scored_cost,
        "edge_cost": check["edge_cost"],
        "num_frequencies_used": check["num_frequencies_used"],
        "normalized_submission": check["normalized_submission"],
        "baseline_cost": baseline_cost,
        "gold_cost": gold_cost,
        "used_baseline_fallback": used_baseline_fallback,
    }
    return gap_pct, feasible, details

def canonical_edge(u: str, v: str) -> tuple[str, str]:
    if u == v:
        raise ValueError(f"self loops are not allowed: {u}")
    return (u, v) if u < v else (v, u)

def ordered_edges(
    edge_keys: set[tuple[str, str]] | tuple[tuple[str, str], ...],
    villages: tuple[str, ...],
) -> list[tuple[str, str]]:
    order = {village: index for index, village in enumerate(villages)}
    return sorted(edge_keys, key=lambda item: (order[item[0]], order[item[1]]))

def active_villages_for_edges(
    terminals: tuple[str, ...],
    edge_keys: set[tuple[str, str]],
    villages: tuple[str, ...],
) -> tuple[str, ...]:
    active = set(terminals)
    for u, v in edge_keys:
        active.add(u)
        active.add(v)
    return tuple(village for village in villages if village in active)

def _require_list_of_strings(instance: dict[str, Any], key: str) -> list[str]:
    value = instance.get(key)
    if not isinstance(value, list) or not all(isinstance(item, str) for item in value):
        raise ValueError(f"instance[{key!r}] must be a list of strings")
    return value

def _require_string_pairs(instance: dict[str, Any], key: str) -> list[list[str]]:
    value = instance.get(key)
    if not isinstance(value, list):
        raise ValueError(f"instance[{key!r}] must be a list")
    normalized: list[list[str]] = []
    for item in value:
        if not isinstance(item, list | tuple) or len(item) != 2:
            raise ValueError(f"instance[{key!r}] must contain two-item pairs")
        left, right = item
        if not isinstance(left, str) or not isinstance(right, str):
            raise ValueError(f"instance[{key!r}] must contain string pairs")
        normalized.append([left, right])
    return normalized

def _require_edges(instance: dict[str, Any]) -> list[dict[str, Any]]:
    value = instance.get("edges")
    if not isinstance(value, list):
        raise ValueError("instance['edges'] must be a list")
    normalized: list[dict[str, Any]] = []
    for item in value:
        if not isinstance(item, dict):
            raise ValueError("instance['edges'] entries must be objects")
        u = item.get("u")
        v = item.get("v")
        cost = item.get("cost")
        if not isinstance(u, str) or not isinstance(v, str):
            raise ValueError("instance['edges'] entries must contain string u/v fields")
        normalized.append({"u": u, "v": v, "cost": int(cost)})
    return normalized

def _evaluate_submission(
    *,
    villages: tuple[str, ...],
    terminals: tuple[str, ...],
    interference_pairs: tuple[tuple[str, str], ...],
    village_order: dict[str, int],
    edge_lookup: dict[tuple[str, str], int],
    submission: dict[str, Any] | None,
    freq_penalty: int,
) -> dict[str, Any]:
    if not isinstance(submission, dict):
        return _failure("submission must be an object")

    raw_edges = submission.get("edges")
    canonical_edges = _normalize_edges(edge_lookup, raw_edges)
    if isinstance(canonical_edges, str):
        return _failure(canonical_edges)

    active = active_villages_for_edges(terminals, canonical_edges, villages)
    if len(canonical_edges) != len(active) - 1:
        return _failure("chosen cable links do not form a tree on the active villages")
    if not _is_connected(active, canonical_edges):
        return _failure("chosen cable links do not connect the active villages")
    if not set(terminals).issubset(active):
        return _failure("all required terminals must be active")

    frequencies = _normalize_frequencies(submission.get("frequencies"))
    if isinstance(frequencies, str):
        return _failure(frequencies)

    missing = [village for village in active if village not in frequencies]
    if missing:
        return _failure(f"missing frequencies for active villages: {missing}")

    active_set = set(active)
    for u, v in interference_pairs:
        if u in active_set and v in active_set and frequencies[u] == frequencies[v]:
            return _failure(f"interference violation between {u} and {v}")

    edge_cost = sum(edge_lookup[edge] for edge in canonical_edges)
    num_frequencies_used = len({frequencies[village] for village in active})
    computed_cost = edge_cost + freq_penalty * num_frequencies_used
    normalized_submission = {
        "edges": [list(edge) for edge in ordered_edges(canonical_edges, villages)],
        "frequencies": {
            village: frequencies[village]
            for village in sorted(active, key=lambda name: village_order[name])
        },
    }
    return {
        "feasible": True,
        "failure_reason": None,
        "computed_cost": computed_cost,
        "edge_cost": edge_cost,
        "num_frequencies_used": num_frequencies_used,
        "normalized_submission": normalized_submission,
    }

def _normalize_edges(
    edge_lookup: dict[tuple[str, str], int],
    raw_edges: Any,
) -> set[tuple[str, str]] | str:
    if not isinstance(raw_edges, list):
        return "edges must be a list"
    canonical: set[tuple[str, str]] = set()
    for item in raw_edges:
        if not isinstance(item, list | tuple) or len(item) != 2:
            return "each edge must be a two-item list"
        try:
            u = str(item[0]).strip()
            v = str(item[1]).strip()
        except Exception:
            return "edges must contain village names"
        edge = canonical_edge(u, v)
        if edge not in edge_lookup:
            return f"unknown cable edge: {edge[0]}-{edge[1]}"
        canonical.add(edge)
    return canonical

def _normalize_frequencies(raw_frequencies: Any) -> dict[str, int] | str:
    if not isinstance(raw_frequencies, dict):
        return "frequencies must be an object"
    normalized: dict[str, int] = {}
    for village, value in raw_frequencies.items():
        try:
            freq = int(value)
        except Exception:
            return f"frequency for {village} is not an integer"
        if freq <= 0:
            return f"frequency for {village} must be positive"
        normalized[str(village).strip()] = freq
    return normalized

def _is_connected(active: tuple[str, ...], edges: set[tuple[str, str]]) -> bool:
    if not active:
        return False
    adjacency = {village: set() for village in active}
    for u, v in edges:
        adjacency[u].add(v)
        adjacency[v].add(u)
    stack = [active[0]]
    seen: set[str] = set()
    while stack:
        village = stack.pop()
        if village in seen:
            continue
        seen.add(village)
        stack.extend(adjacency[village] - seen)
    return seen == set(active)

def _failure(reason: str) -> dict[str, Any]:
    return {
        "feasible": False,
        "failure_reason": reason,
        "computed_cost": None,
        "edge_cost": None,
        "num_frequencies_used": None,
        "normalized_submission": None,
    }

# ──────────────────────────────────────────────────────────────────────
# verifiers/tsp.py
# ──────────────────────────────────────────────────────────────────────

from math import hypot
from typing import Any

INVALID_GAP_PCT = 100.0

def tsp_verify(instance: dict[str, Any], submission: dict[str, Any] | list[int] | tuple[int, ...]) -> tuple[float, bool, dict[str, Any]]:
    """BEST_GUESS JSON schema: {"tour": [0, 7, 3, ...]} with every city index exactly once."""

    try:
        coords = _parse_coords(instance.get("coords"))
        optimal_length = _read_positive_float(instance, "optimal_length")
    except (TypeError, ValueError) as exc:
        details = {"failure_reason": f"invalid instance: {exc}"}
        return INVALID_GAP_PCT, False, details

    baseline_length = _read_optional_float(instance, "baseline_length")
    raw_tour = submission.get("tour") if isinstance(submission, dict) else submission
    normalized_tour, failure_reason = _normalize_submission(coords, raw_tour)
    if failure_reason is not None:
        details = {
            "failure_reason": failure_reason,
            "baseline_length": baseline_length,
            "optimal_length": optimal_length,
        }
        return INVALID_GAP_PCT, False, details

    computed_length = _tour_length(coords, normalized_tour)
    gap_pct = 100.0 * max(0.0, computed_length - optimal_length) / optimal_length
    details = {
        "computed_length": computed_length,
        "baseline_length": baseline_length,
        "optimal_length": optimal_length,
        "normalized_tour": list(normalized_tour),
    }
    return gap_pct, True, details

def _parse_coords(raw_coords: Any) -> tuple[tuple[int, int], ...]:
    if not isinstance(raw_coords, list):
        raise TypeError("coords must be a list")
    coords: list[tuple[int, int]] = []
    for item in raw_coords:
        if not isinstance(item, list | tuple) or len(item) != 2:
            raise TypeError("each coordinate must be a two-item list")
        x = int(item[0])
        y = int(item[1])
        coords.append((x, y))
    if not coords:
        raise ValueError("coords must not be empty")
    return tuple(coords)

def _normalize_submission(
    coords: tuple[tuple[int, int], ...],
    raw_tour: Any,
) -> tuple[tuple[int, ...], str | None]:
    if not isinstance(raw_tour, list | tuple):
        return tuple(), "submission must provide tour as a list of city indices"

    n = len(coords)
    if len(raw_tour) != n:
        return tuple(), f"tour must contain exactly {n} cities"

    normalized: list[int] = []
    seen: set[int] = set()
    for item in raw_tour:
        try:
            city = int(item)
        except Exception:
            return tuple(), "tour items must be integers"
        if not 0 <= city < n:
            return tuple(), f"city index {city} is out of range"
        if city in seen:
            return tuple(), f"city {city} appears more than once"
        normalized.append(city)
        seen.add(city)

    return _normalize_tour(normalized), None

def _normalize_tour(tour: list[int] | tuple[int, ...]) -> tuple[int, ...]:
    start_index = tour.index(0)
    rotated = list(tour[start_index:]) + list(tour[:start_index])
    reversed_rotated = [rotated[0]] + list(reversed(rotated[1:]))
    return tuple(rotated) if tuple(rotated) <= tuple(reversed_rotated) else tuple(reversed_rotated)

def _tour_length(coords: tuple[tuple[int, int], ...], tour: tuple[int, ...]) -> float:
    total = 0.0
    for idx, city in enumerate(tour):
        nxt = tour[(idx + 1) % len(tour)]
        total += _distance(coords, city, nxt)
    return total

def _distance(coords: tuple[tuple[int, int], ...], a: int, b: int) -> float:
    ax, ay = coords[a]
    bx, by = coords[b]
    return hypot(ax - bx, ay - by)

def _read_positive_float(instance: dict[str, Any], key: str) -> float:
    value = float(instance[key])
    if value <= 0.0:
        raise ValueError(f"{key} must be positive")
    return value

def _read_optional_float(instance: dict[str, Any], key: str) -> float | None:
    value = instance.get(key)
    if value is None:
        return None
    return float(value)

# ──────────────────────────────────────────────────────────────────────
# verifiers/ve.py
# ──────────────────────────────────────────────────────────────────────

"""BEST_GUESS JSON schema:
{
  "p_query_given_evidence": <float strictly between 0 and 1>,
  "ordering_used": ["X1", "X2", "..."],  # every eliminable variable exactly once
  "peak_factor_size_self_report": <positive int>
}
"""
from dataclasses import dataclass
from itertools import combinations
import math
from random import Random
from typing import Any, Iterable, Sequence

RANDOM_ORDER_SAMPLES = 1000
DEFAULT_VARIABLE_SIZES = (22, 26, 28)
DIFFICULTY_TO_VARIABLES = {
    "medium": 22,
    "hard": 26,
}
FAILURE_GAP_PCT = 100.0
MIN_POSITIVE = 1e-12

@dataclass(frozen=True)
class VariableSpec:
    name: str
    parents: tuple[str, ...]
    cpt_true_probs: tuple[float, ...]

@dataclass(frozen=True)
class Factor:
    scope: tuple[str, ...]
    values: tuple[float, ...]

@dataclass(frozen=True)
class OrderingSummary:
    name: str
    ordering: tuple[str, ...]
    peak_factor_size: int

@dataclass(frozen=True)
class GuessVerification:
    is_valid: bool
    failure_reason: str | None
    p_query_given_evidence: float | None
    ordering_used: tuple[str, ...] | None
    peak_factor_size_self_report: int | None
    true_peak_factor_size: int | None
    gap_nats: float | None

@dataclass(frozen=True)
class GeneratedCandidate:
    total_variables: int
    variables: tuple[VariableSpec, ...]
    query_variable: str
    evidence: dict[str, int]
    hidden_clusters: dict[str, tuple[str, ...]]
    bridge_variables: tuple[str, ...]
    decoy_variable: str

@dataclass(frozen=True)
class BayesianVEInstance:
    seed: int
    difficulty: str
    requested_total_variables: int
    total_variables: int
    variables: tuple[VariableSpec, ...]
    query_variable: str
    evidence: dict[str, int]
    eliminable_variables: tuple[str, ...]
    exact_posterior: float
    baseline_ordering: tuple[str, ...]
    baseline_peak_factor_size: int
    gold_ordering: tuple[str, ...]
    gold_peak_factor_size: int
    gold_source: str
    heuristic_results: tuple[OrderingSummary, ...]
    random_search_best: OrderingSummary
    min_fill_peak_factor_size: int
    problem_statement: str
    hidden_clusters: dict[str, tuple[str, ...]]
    bridge_variables: tuple[str, ...]
    decoy_variable: str
    attempt_index: int
    size_escalated: bool
    scale_note: str | None

def build_instance_payload(seed: int, difficulty: str) -> dict[str, Any]:
    requested_total_variables = _difficulty_to_requested_total_variables(difficulty)
    instance = build_instance(
        seed=seed,
        difficulty=difficulty,
        requested_total_variables=requested_total_variables,
    )
    return _instance_to_payload(instance)

def ve_verify(instance: dict[str, Any], submission: dict[str, Any]) -> tuple[float, bool, dict[str, Any]]:
    try:
        ve_instance = _instance_from_payload(instance)
    except ValueError as exc:
        return FAILURE_GAP_PCT, False, {"failure_reason": f"invalid instance: {exc}"}

    verification = verify_best_guess(ve_instance, submission)
    if not verification.is_valid:
        return FAILURE_GAP_PCT, False, {"failure_reason": verification.failure_reason}

    submitted_probability = float(verification.p_query_given_evidence)
    gap_pct = _posterior_gap_pct(ve_instance.exact_posterior, submitted_probability)
    return (
        gap_pct,
        True,
        {
            "submitted_probability": submitted_probability,
            "exact_posterior": ve_instance.exact_posterior,
            "absolute_probability_error": abs(submitted_probability - ve_instance.exact_posterior),
            "gap_pct": gap_pct,
            "gap_nats": verification.gap_nats,
            "ordering_used": list(verification.ordering_used or ()),
            "submitted_peak_factor_size": verification.peak_factor_size_self_report,
            "true_peak_factor_size": verification.true_peak_factor_size,
            "gold_ordering": list(ve_instance.gold_ordering),
            "gold_peak_factor_size": ve_instance.gold_peak_factor_size,
            "gold_source": ve_instance.gold_source,
            "peak_factor_size_matches": (
                verification.peak_factor_size_self_report == verification.true_peak_factor_size
            ),
        },
    )

def build_instance(
    *,
    seed: int,
    difficulty: str,
    requested_total_variables: int = 22,
    max_generation_attempts: int = 16,
    random_order_samples: int = RANDOM_ORDER_SAMPLES,
) -> BayesianVEInstance:
    size_queue = [requested_total_variables]
    size_queue.extend(
        size for size in DEFAULT_VARIABLE_SIZES if size > requested_total_variables and size not in size_queue
    )

    best_candidate: BayesianVEInstance | None = None
    best_score: tuple[int, int, float] | None = None
    last_reason = "no candidate generated"
    scale_note: str | None = None

    for size_index, total_variables in enumerate(size_queue):
        best_for_size: BayesianVEInstance | None = None
        best_for_size_score: tuple[int, int, float] | None = None
        any_hard_enough = False

        for attempt_index in range(max_generation_attempts):
            candidate_seed = seed + size_index * 100_000 + attempt_index * 10_007
            candidate = _generate_candidate(seed=candidate_seed, total_variables=total_variables)
            instance = _finalize_candidate(
                seed=seed,
                difficulty=difficulty,
                requested_total_variables=requested_total_variables,
                candidate=candidate,
                attempt_index=attempt_index,
                random_order_samples=random_order_samples,
                scale_note=scale_note,
            )
            score = _candidate_score(instance)
            if best_candidate is None or score > best_score:
                best_candidate = instance
                best_score = score
            if best_for_size is None or score > best_for_size_score:
                best_for_size = instance
                best_for_size_score = score
            if total_variables == 22 and instance.min_fill_peak_factor_size <= 4:
                last_reason = (
                    f"attempt {attempt_index} at 22 vars still looked easy "
                    f"(min-fill peak factor size {instance.min_fill_peak_factor_size})"
                )
                continue
            any_hard_enough = True

        if total_variables == 22 and not any_hard_enough:
            scale_note = "All 22-variable attempts had min-fill peak factor size <= 4, so the spike escalated."
            last_reason = scale_note
            continue
        if best_for_size is not None:
            return best_for_size

    if best_candidate is not None:
        return best_candidate
    raise RuntimeError(f"failed to generate a Bayesian VE instance: {last_reason}")

def verify_best_guess(instance: BayesianVEInstance, best_guess: dict[str, Any]) -> GuessVerification:
    if not isinstance(best_guess, dict):
        return GuessVerification(False, "BEST_GUESS must be a JSON object", None, None, None, None, None)

    try:
        probability = float(best_guess["p_query_given_evidence"])
    except Exception:
        return GuessVerification(False, "p_query_given_evidence must be numeric", None, None, None, None, None)
    if not 0.0 < probability < 1.0:
        return GuessVerification(
            False,
            "p_query_given_evidence must lie strictly between 0 and 1",
            None,
            None,
            None,
            None,
            None,
        )

    ordering_used = best_guess.get("ordering_used")
    normalized_order = normalize_elimination_order(instance, ordering_used)
    if normalized_order is None:
        return GuessVerification(
            False,
            "ordering_used must list every eliminable variable exactly once and exclude query/evidence variables",
            None,
            None,
            None,
            None,
            None,
        )

    try:
        self_reported_peak = int(best_guess["peak_factor_size_self_report"])
    except Exception:
        return GuessVerification(
            False,
            "peak_factor_size_self_report must be an integer",
            None,
            None,
            None,
            None,
            None,
        )
    if self_reported_peak < 1:
        return GuessVerification(
            False,
            "peak_factor_size_self_report must be positive",
            None,
            None,
            None,
            None,
            None,
        )

    _, true_peak = evaluate_exact_probability(instance, normalized_order)
    gap_nats = abs(math.log(instance.exact_posterior) - math.log(probability))
    return GuessVerification(
        is_valid=True,
        failure_reason=None,
        p_query_given_evidence=probability,
        ordering_used=normalized_order,
        peak_factor_size_self_report=self_reported_peak,
        true_peak_factor_size=true_peak,
        gap_nats=gap_nats,
    )

def normalize_elimination_order(
    instance: BayesianVEInstance,
    ordering: Sequence[Any] | None,
) -> tuple[str, ...] | None:
    if not isinstance(ordering, Sequence) or isinstance(ordering, (str, bytes)):
        return None
    names = tuple(str(value).strip() for value in ordering)
    if not all(names):
        return None
    if len(names) != len(instance.eliminable_variables):
        return None
    if set(names) != set(instance.eliminable_variables):
        return None
    if len(set(names)) != len(names):
        return None
    return names

def render_problem(instance: BayesianVEInstance) -> str:
    evidence_terms = ", ".join(f"{name}={value}" for name, value in instance.evidence.items())
    lines = [
        "You are doing exact inference in a Bayesian network with binary variables in {0, 1}.",
        f"Task: report P({instance.query_variable}=1 | {evidence_terms}).",
        "All CPT rows below give P(variable=1 | parents); P(variable=0 | parents) = 1 minus that value.",
        "When you report ordering_used, list every eliminable variable exactly once and do not include the query variable or any evidence variable.",
        "Interpret peak_factor_size as the largest intermediate factor scope size, counted as the number of variables in that factor before summing out the eliminated variable.",
        "",
        f"Variables: {', '.join(variable.name for variable in instance.variables)}",
        f"Query variable: {instance.query_variable}",
        f"Evidence: {evidence_terms}",
        f"Eliminable variables ({len(instance.eliminable_variables)}): {', '.join(instance.eliminable_variables)}",
        "",
        "Bayesian network (topological order):",
    ]
    for variable in instance.variables:
        parent_text = ", ".join(variable.parents) if variable.parents else "none"
        lines.append(f"- {variable.name}: parents = [{parent_text}]")
        if not variable.parents:
            lines.append(f"  P({variable.name}=1) = {_fmt_prob(variable.cpt_true_probs[0])}")
            continue
        for assignment_index, probability in enumerate(variable.cpt_true_probs):
            assignment = _bits_from_index(assignment_index, len(variable.parents))
            parent_terms = ", ".join(
                f"{parent}={value}" for parent, value in zip(variable.parents, assignment, strict=True)
            )
            lines.append(f"  if {parent_terms} -> P({variable.name}=1) = {_fmt_prob(probability)}")
    return "\n".join(lines)

def evaluate_exact_probability(
    instance: BayesianVEInstance,
    ordering: Sequence[str],
) -> tuple[float, int]:
    normalized_order = normalize_elimination_order(instance, ordering)
    if normalized_order is None:
        raise ValueError("invalid elimination ordering")

    factors = list(_build_conditioned_factors(instance.variables, instance.evidence))
    peak_factor_size = max((len(factor.scope) for factor in factors if factor.scope), default=1)

    for variable_name in normalized_order:
        involved = [factor for factor in factors if variable_name in factor.scope]
        untouched = [factor for factor in factors if variable_name not in factor.scope]
        if not involved:
            factors = untouched
            continue
        product_factor = _multiply_all(involved)
        peak_factor_size = max(peak_factor_size, len(product_factor.scope))
        reduced = _sum_out(product_factor, variable_name)
        if reduced.scope or not math.isclose(reduced.values[0], 1.0, rel_tol=0.0, abs_tol=1e-12):
            untouched.append(reduced)
        factors = untouched

    final_factor = _multiply_all(factors)
    if instance.query_variable not in final_factor.scope:
        raise RuntimeError("query variable disappeared during elimination")
    probability_zero = _factor_probability_for_assignment(final_factor, {instance.query_variable: 0})
    probability_one = _factor_probability_for_assignment(final_factor, {instance.query_variable: 1})
    normalizer = probability_zero + probability_one
    if normalizer <= 0.0:
        raise RuntimeError("invalid posterior normalizer")
    return probability_one / normalizer, peak_factor_size

def evaluate_ordering_peak_from_scopes(
    initial_scopes: Sequence[frozenset[str]],
    ordering: Sequence[str],
) -> int:
    scopes = [set(scope) for scope in initial_scopes if scope]
    peak_factor_size = max((len(scope) for scope in scopes), default=1)
    for variable_name in ordering:
        touched = [scope for scope in scopes if variable_name in scope]
        untouched = [scope for scope in scopes if variable_name not in scope]
        if not touched:
            scopes = untouched
            continue
        merged_scope = set().union(*touched)
        peak_factor_size = max(peak_factor_size, len(merged_scope))
        next_scope = merged_scope - {variable_name}
        if next_scope:
            untouched.append(next_scope)
        scopes = untouched
    return peak_factor_size

def _finalize_candidate(
    *,
    seed: int,
    difficulty: str,
    requested_total_variables: int,
    candidate: GeneratedCandidate,
    attempt_index: int,
    random_order_samples: int,
    scale_note: str | None,
) -> BayesianVEInstance:
    eliminable_variables = tuple(
        variable.name
        for variable in candidate.variables
        if variable.name != candidate.query_variable and variable.name not in candidate.evidence
    )
    initial_scopes = tuple(
        frozenset(factor.scope)
        for factor in _build_conditioned_factors(candidate.variables, candidate.evidence)
        if factor.scope
    )

    heuristic_orders = (
        OrderingSummary(
            name="baseline",
            ordering=tuple(sorted(eliminable_variables)),
            peak_factor_size=evaluate_ordering_peak_from_scopes(initial_scopes, tuple(sorted(eliminable_variables))),
        ),
        OrderingSummary(
            name="min_neighbors",
            ordering=_build_greedy_order(initial_scopes, eliminable_variables, rule="min_neighbors"),
            peak_factor_size=0,
        ),
        OrderingSummary(
            name="min_weight",
            ordering=_build_greedy_order(initial_scopes, eliminable_variables, rule="min_weight"),
            peak_factor_size=0,
        ),
        OrderingSummary(
            name="min_fill",
            ordering=_build_greedy_order(initial_scopes, eliminable_variables, rule="min_fill"),
            peak_factor_size=0,
        ),
    )
    realized_heuristics: list[OrderingSummary] = []
    for summary in heuristic_orders:
        peak = summary.peak_factor_size or evaluate_ordering_peak_from_scopes(initial_scopes, summary.ordering)
        realized_heuristics.append(
            OrderingSummary(
                name=summary.name,
                ordering=summary.ordering,
                peak_factor_size=peak,
            )
        )

    rng = Random(seed + candidate.total_variables * 997 + attempt_index * 13)
    random_best_ordering = tuple(eliminable_variables)
    random_best_peak = evaluate_ordering_peak_from_scopes(initial_scopes, random_best_ordering)
    for _ in range(random_order_samples):
        proposal = list(eliminable_variables)
        rng.shuffle(proposal)
        peak = evaluate_ordering_peak_from_scopes(initial_scopes, proposal)
        if peak < random_best_peak:
            random_best_ordering = tuple(proposal)
            random_best_peak = peak
    random_search_best = OrderingSummary(
        name="random_best_of_1000",
        ordering=random_best_ordering,
        peak_factor_size=random_best_peak,
    )

    gold_candidates = list(realized_heuristics[1:]) + [random_search_best]
    gold = min(gold_candidates, key=lambda summary: (summary.peak_factor_size, summary.name, summary.ordering))
    stub = BayesianVEInstance(
        seed=seed,
        difficulty=difficulty,
        requested_total_variables=requested_total_variables,
        total_variables=candidate.total_variables,
        variables=candidate.variables,
        query_variable=candidate.query_variable,
        evidence=candidate.evidence,
        eliminable_variables=eliminable_variables,
        exact_posterior=0.0,
        baseline_ordering=realized_heuristics[0].ordering,
        baseline_peak_factor_size=realized_heuristics[0].peak_factor_size,
        gold_ordering=gold.ordering,
        gold_peak_factor_size=gold.peak_factor_size,
        gold_source=gold.name,
        heuristic_results=tuple(realized_heuristics[1:]),
        random_search_best=random_search_best,
        min_fill_peak_factor_size=next(
            summary.peak_factor_size for summary in realized_heuristics if summary.name == "min_fill"
        ),
        problem_statement="",
        hidden_clusters=candidate.hidden_clusters,
        bridge_variables=candidate.bridge_variables,
        decoy_variable=candidate.decoy_variable,
        attempt_index=attempt_index,
        size_escalated=candidate.total_variables != requested_total_variables,
        scale_note=scale_note,
    )
    exact_posterior, _ = evaluate_exact_probability(stub, gold.ordering)
    instance_without_render = BayesianVEInstance(
        seed=seed,
        difficulty=difficulty,
        requested_total_variables=requested_total_variables,
        total_variables=candidate.total_variables,
        variables=candidate.variables,
        query_variable=candidate.query_variable,
        evidence=candidate.evidence,
        eliminable_variables=eliminable_variables,
        exact_posterior=exact_posterior,
        baseline_ordering=realized_heuristics[0].ordering,
        baseline_peak_factor_size=realized_heuristics[0].peak_factor_size,
        gold_ordering=gold.ordering,
        gold_peak_factor_size=gold.peak_factor_size,
        gold_source=gold.name,
        heuristic_results=tuple(realized_heuristics[1:]),
        random_search_best=random_search_best,
        min_fill_peak_factor_size=next(
            summary.peak_factor_size for summary in realized_heuristics if summary.name == "min_fill"
        ),
        problem_statement="",
        hidden_clusters=candidate.hidden_clusters,
        bridge_variables=candidate.bridge_variables,
        decoy_variable=candidate.decoy_variable,
        attempt_index=attempt_index,
        size_escalated=candidate.total_variables != requested_total_variables,
        scale_note=scale_note,
    )
    return BayesianVEInstance(
        seed=seed,
        difficulty=difficulty,
        requested_total_variables=requested_total_variables,
        total_variables=candidate.total_variables,
        variables=candidate.variables,
        query_variable=candidate.query_variable,
        evidence=candidate.evidence,
        eliminable_variables=eliminable_variables,
        exact_posterior=exact_posterior,
        baseline_ordering=realized_heuristics[0].ordering,
        baseline_peak_factor_size=realized_heuristics[0].peak_factor_size,
        gold_ordering=gold.ordering,
        gold_peak_factor_size=gold.peak_factor_size,
        gold_source=gold.name,
        heuristic_results=tuple(realized_heuristics[1:]),
        random_search_best=random_search_best,
        min_fill_peak_factor_size=next(
            summary.peak_factor_size for summary in realized_heuristics if summary.name == "min_fill"
        ),
        problem_statement=render_problem(instance_without_render),
        hidden_clusters=candidate.hidden_clusters,
        bridge_variables=candidate.bridge_variables,
        decoy_variable=candidate.decoy_variable,
        attempt_index=attempt_index,
        size_escalated=candidate.total_variables != requested_total_variables,
        scale_note=scale_note,
    )

def _candidate_score(instance: BayesianVEInstance) -> tuple[int, int, float]:
    heuristic_peaks = [summary.peak_factor_size for summary in instance.heuristic_results]
    spread = max(heuristic_peaks) - min(heuristic_peaks)
    baseline_advantage = instance.baseline_peak_factor_size - instance.gold_peak_factor_size
    posterior_balance = -abs(instance.exact_posterior - 0.5)
    return (baseline_advantage, spread, posterior_balance)

def _generate_candidate(*, seed: int, total_variables: int) -> GeneratedCandidate:
    rng = Random(seed)
    cluster_sizes = _cluster_sizes(total_variables - 3)
    hidden_clusters = {
        "A": tuple(f"A{index}" for index in range(1, cluster_sizes[0] + 1)),
        "B": tuple(f"B{index}" for index in range(1, cluster_sizes[1] + 1)),
        "C": tuple(f"C{index}" for index in range(1, cluster_sizes[2] + 1)),
    }
    bridge_variables = ("BR01", "BR12")
    decoy_variable = "D0"

    early_clusters = {name: values[: max(3, len(values) // 2)] for name, values in hidden_clusters.items()}
    late_clusters = {name: values[len(early_clusters[name]) :] for name, values in hidden_clusters.items()}

    topological_order = (
        list(hidden_clusters["A"][: len(early_clusters["A"])])
        + list(hidden_clusters["B"][: len(early_clusters["B"])])
        + list(hidden_clusters["C"][: len(early_clusters["C"])])
        + [decoy_variable, *bridge_variables]
        + list(late_clusters["A"])
        + list(late_clusters["B"])
        + list(late_clusters["C"])
    )

    parent_map: dict[str, tuple[str, ...]] = {}
    position = {name: index for index, name in enumerate(topological_order)}

    for cluster_name in ("A", "B", "C"):
        early_nodes = early_clusters[cluster_name]
        for index, variable_name in enumerate(early_nodes):
            parents: list[str] = []
            if index >= 1:
                parents.append(early_nodes[index - 1])
            if index >= 2:
                parents.append(early_nodes[index - 2])
            if cluster_name == "B" and index == 0:
                parents.append(hidden_clusters["A"][min(1, len(hidden_clusters["A"]) - 1)])
            if cluster_name == "C" and index == 0:
                parents.append(hidden_clusters["B"][min(1, len(hidden_clusters["B"]) - 1)])
            parent_map[variable_name] = tuple(dict.fromkeys(parents))

    parent_map[decoy_variable] = (
        early_clusters["A"][-1],
        early_clusters["B"][-1],
        early_clusters["C"][-1],
    )
    parent_map["BR01"] = (
        early_clusters["A"][-2],
        early_clusters["B"][-2],
    )
    parent_map["BR12"] = (
        early_clusters["B"][-1],
        early_clusters["C"][-2],
    )

    for cluster_name in ("A", "B", "C"):
        late_nodes = late_clusters[cluster_name]
        all_cluster_nodes = hidden_clusters[cluster_name]
        for index, variable_name in enumerate(late_nodes):
            earlier_same_cluster = [name for name in all_cluster_nodes if position[name] < position[variable_name]]
            parents = list(earlier_same_cluster[-2:]) if len(earlier_same_cluster) >= 2 else list(earlier_same_cluster)
            if cluster_name == "A":
                if index in {0, len(late_nodes) - 1}:
                    parents.append(decoy_variable)
            elif cluster_name == "B":
                if index == 0:
                    parents.append("BR01")
                if index == 1:
                    parents.append(decoy_variable)
                if index == len(late_nodes) - 1:
                    parents.extend([decoy_variable, "BR12"])
            else:
                if index == 0:
                    parents.append("BR12")
                if index in {0, len(late_nodes) - 1}:
                    parents.append(decoy_variable)
            if cluster_name == "B" and index < len(hidden_clusters["A"]):
                parents.append(hidden_clusters["A"][min(len(hidden_clusters["A"]) - 1, index + 1)])
            if cluster_name == "C" and index < len(hidden_clusters["B"]):
                parents.append(hidden_clusters["B"][min(len(hidden_clusters["B"]) - 1, index + 1)])
            parent_map[variable_name] = tuple(dict.fromkeys(parents[-3:]))

    variables = tuple(
        VariableSpec(
            name=variable_name,
            parents=parent_map.get(variable_name, ()),
            cpt_true_probs=_generate_cpt(
                rng=rng,
                variable_name=variable_name,
                parents=parent_map.get(variable_name, ()),
            ),
        )
        for variable_name in topological_order
    )

    query_variable = late_clusters["B"][-1]
    evidence_candidates = [
        late_clusters["A"][-1],
        hidden_clusters["B"][0],
        late_clusters["B"][0],
        late_clusters["C"][-1],
        hidden_clusters["C"][1],
    ]
    evidence = {
        variable_name: rng.randint(0, 1)
        for variable_name in evidence_candidates
        if variable_name != query_variable
    }
    return GeneratedCandidate(
        total_variables=total_variables,
        variables=variables,
        query_variable=query_variable,
        evidence=evidence,
        hidden_clusters=hidden_clusters,
        bridge_variables=bridge_variables,
        decoy_variable=decoy_variable,
    )

def _cluster_sizes(core_variable_count: int) -> tuple[int, int, int]:
    base = core_variable_count // 3
    remainder = core_variable_count - 3 * base
    return (base, base + remainder, base)

def _generate_cpt(*, rng: Random, variable_name: str, parents: Sequence[str]) -> tuple[float, ...]:
    if not parents:
        return (_rounded_probability(0.35 + 0.30 * rng.random()),)

    base = rng.uniform(-0.9, 0.9)
    weights = []
    for parent_name in parents:
        magnitude = rng.uniform(0.55, 1.35)
        sign = -1.0 if rng.random() < 0.45 else 1.0
        if parent_name in {"D0", "BR01", "BR12"} or variable_name in {"D0", "BR01", "BR12"}:
            magnitude += 0.15
        weights.append(sign * magnitude)

    probabilities = []
    for assignment_index in range(1 << len(parents)):
        assignment = _bits_from_index(assignment_index, len(parents))
        logit = base
        for bit, weight in zip(assignment, weights, strict=True):
            logit += weight * (1.0 if bit else -1.0)
        probability = 1.0 / (1.0 + math.exp(-logit))
        probabilities.append(_rounded_probability(probability))
    return tuple(probabilities)

def _rounded_probability(probability: float) -> float:
    clipped = min(0.95, max(0.05, probability))
    return round(clipped, 3)

def _build_conditioned_factors(
    variables: Sequence[VariableSpec],
    evidence: dict[str, int],
) -> tuple[Factor, ...]:
    factors: list[Factor] = []
    for variable in variables:
        reduced_scope = [parent for parent in variable.parents if parent not in evidence]
        include_variable = variable.name not in evidence
        if include_variable:
            reduced_scope.append(variable.name)
        values: list[float] = []
        for assignment_index in range(1 << len(reduced_scope)):
            reduced_assignment = _bits_from_index(assignment_index, len(reduced_scope))
            assignment_map = dict(zip(reduced_scope, reduced_assignment, strict=True))
            parent_assignment = tuple(
                evidence[parent_name] if parent_name in evidence else assignment_map[parent_name]
                for parent_name in variable.parents
            )
            variable_value = evidence[variable.name] if variable.name in evidence else assignment_map[variable.name]
            p_true = variable.cpt_true_probs[_index_from_bits(parent_assignment)]
            values.append(p_true if variable_value == 1 else 1.0 - p_true)
        factors.append(Factor(scope=tuple(reduced_scope), values=tuple(values)))
    return tuple(factors)

def _build_greedy_order(
    initial_scopes: Sequence[frozenset[str]],
    eliminable_variables: Sequence[str],
    *,
    rule: str,
) -> tuple[str, ...]:
    remaining = list(eliminable_variables)
    scopes = [set(scope) for scope in initial_scopes if scope]
    order: list[str] = []

    while remaining:
        adjacency = _adjacency_from_scopes(scopes)
        incident_scopes = {name: [scope for scope in scopes if name in scope] for name in remaining}

        def score(name: str) -> tuple[Any, ...]:
            neighbors = adjacency.get(name, set())
            mass = sum(1 << len(scope) for scope in incident_scopes[name]) or 1
            if rule == "min_neighbors":
                return (len(neighbors), mass, name)
            if rule == "min_weight":
                return (mass, len(neighbors), name)
            if rule == "min_fill":
                fill_edges = 0
                neighbor_list = sorted(neighbors)
                for left, right in combinations(neighbor_list, 2):
                    if right not in adjacency.get(left, set()):
                        fill_edges += 1
                return (fill_edges, len(neighbors), mass, name)
            raise ValueError(f"unknown rule: {rule}")

        choice = min(remaining, key=score)
        order.append(choice)
        remaining.remove(choice)

        touched = [scope for scope in scopes if choice in scope]
        untouched = [scope for scope in scopes if choice not in scope]
        merged = set().union(*touched) if touched else {choice}
        next_scope = merged - {choice}
        if next_scope:
            untouched.append(next_scope)
        scopes = untouched

    return tuple(order)

def _adjacency_from_scopes(scopes: Iterable[set[str]]) -> dict[str, set[str]]:
    adjacency: dict[str, set[str]] = {}
    for scope in scopes:
        for variable_name in scope:
            adjacency.setdefault(variable_name, set())
        for left, right in combinations(scope, 2):
            adjacency[left].add(right)
            adjacency[right].add(left)
    return adjacency

def _multiply_all(factors: Sequence[Factor]) -> Factor:
    if not factors:
        return Factor(scope=(), values=(1.0,))
    result = factors[0]
    for factor in factors[1:]:
        result = _multiply(result, factor)
    return result

def _multiply(left: Factor, right: Factor) -> Factor:
    scope = tuple(dict.fromkeys((*left.scope, *right.scope)))
    values: list[float] = []
    for assignment_index in range(1 << len(scope)):
        assignment = _bits_from_index(assignment_index, len(scope))
        assignment_map = dict(zip(scope, assignment, strict=True))
        values.append(
            _factor_probability_for_assignment(left, assignment_map)
            * _factor_probability_for_assignment(right, assignment_map)
        )
    return Factor(scope=scope, values=tuple(values))

def _sum_out(factor: Factor, variable_name: str) -> Factor:
    if variable_name not in factor.scope:
        return factor
    reduced_scope = tuple(name for name in factor.scope if name != variable_name)
    values: list[float] = []
    for assignment_index in range(1 << len(reduced_scope)):
        assignment = _bits_from_index(assignment_index, len(reduced_scope))
        assignment_map = dict(zip(reduced_scope, assignment, strict=True))
        total = 0.0
        for value in (0, 1):
            total += _factor_probability_for_assignment(
                factor,
                {**assignment_map, variable_name: value},
            )
        values.append(total)
    return Factor(scope=reduced_scope, values=tuple(values))

def _factor_probability_for_assignment(factor: Factor, assignment_map: dict[str, int]) -> float:
    if not factor.scope:
        return factor.values[0]
    bits = tuple(int(assignment_map[name]) for name in factor.scope)
    return factor.values[_index_from_bits(bits)]

def _bits_from_index(index: int, width: int) -> tuple[int, ...]:
    return tuple((index >> shift) & 1 for shift in reversed(range(width)))

def _index_from_bits(bits: Sequence[int]) -> int:
    index = 0
    for bit in bits:
        index = (index << 1) | int(bit)
    return index

def _fmt_prob(probability: float) -> str:
    return f"{probability:.3f}"

def _difficulty_to_requested_total_variables(difficulty: str) -> int:
    normalized = difficulty.strip().lower()
    if normalized not in DIFFICULTY_TO_VARIABLES:
        raise ValueError(f"unsupported difficulty: {difficulty}")
    return DIFFICULTY_TO_VARIABLES[normalized]

def _instance_to_payload(instance: BayesianVEInstance) -> dict[str, Any]:
    return {
        "seed": instance.seed,
        "difficulty": instance.difficulty,
        "requested_total_variables": instance.requested_total_variables,
        "total_variables": instance.total_variables,
        "variables": [
            {
                "name": variable.name,
                "parents": list(variable.parents),
                "cpt_true_probs": list(variable.cpt_true_probs),
            }
            for variable in instance.variables
        ],
        "query_variable": instance.query_variable,
        "evidence": dict(instance.evidence),
        "eliminable_variables": list(instance.eliminable_variables),
        "exact_posterior": instance.exact_posterior,
        "baseline_ordering": list(instance.baseline_ordering),
        "baseline_peak_factor_size": instance.baseline_peak_factor_size,
        "gold_ordering": list(instance.gold_ordering),
        "gold_peak_factor_size": instance.gold_peak_factor_size,
        "gold_source": instance.gold_source,
        "heuristic_results": [
            {
                "name": summary.name,
                "ordering": list(summary.ordering),
                "peak_factor_size": summary.peak_factor_size,
            }
            for summary in instance.heuristic_results
        ],
        "random_search_best": {
            "name": instance.random_search_best.name,
            "ordering": list(instance.random_search_best.ordering),
            "peak_factor_size": instance.random_search_best.peak_factor_size,
        },
        "min_fill_peak_factor_size": instance.min_fill_peak_factor_size,
        "problem_statement": instance.problem_statement,
        "hidden_clusters": {
            cluster_name: list(variable_names)
            for cluster_name, variable_names in instance.hidden_clusters.items()
        },
        "bridge_variables": list(instance.bridge_variables),
        "decoy_variable": instance.decoy_variable,
        "attempt_index": instance.attempt_index,
        "size_escalated": instance.size_escalated,
        "scale_note": instance.scale_note,
    }

def _instance_from_payload(payload: dict[str, Any]) -> BayesianVEInstance:
    if not isinstance(payload, dict):
        raise ValueError("instance must be a dict")

    variables_payload = payload.get("variables")
    if not isinstance(variables_payload, list) or not variables_payload:
        raise ValueError("variables must be a non-empty list")
    variables = tuple(_variable_from_payload(item) for item in variables_payload)

    query_variable = _required_str(payload.get("query_variable"), "query_variable")
    evidence = _int_mapping(payload.get("evidence"), "evidence")
    eliminable_variables = payload.get("eliminable_variables")
    if eliminable_variables is None:
        eliminable_variables_tuple = tuple(
            variable.name for variable in variables if variable.name != query_variable and variable.name not in evidence
        )
    else:
        eliminable_variables_tuple = _string_tuple(eliminable_variables, "eliminable_variables")

    requested_total_variables = _optional_int(payload.get("requested_total_variables"), 22, "requested_total_variables")
    total_variables = _optional_int(payload.get("total_variables"), len(variables), "total_variables")
    seed = _optional_int(payload.get("seed"), 0, "seed")
    difficulty = str(payload.get("difficulty", "medium"))

    initial_scopes = tuple(
        frozenset(factor.scope)
        for factor in _build_conditioned_factors(variables, evidence)
        if factor.scope
    )

    baseline_ordering = tuple(sorted(eliminable_variables_tuple))
    if "baseline_ordering" in payload:
        baseline_ordering = _string_tuple(payload["baseline_ordering"], "baseline_ordering")
    baseline_peak = payload.get("baseline_peak_factor_size")
    if baseline_peak is None:
        baseline_peak_factor_size = evaluate_ordering_peak_from_scopes(initial_scopes, baseline_ordering)
    else:
        baseline_peak_factor_size = _optional_int(baseline_peak, 0, "baseline_peak_factor_size")

    heuristic_payload = payload.get("heuristic_results")
    if heuristic_payload is None:
        heuristic_results = (
            OrderingSummary(
                name="min_neighbors",
                ordering=_build_greedy_order(initial_scopes, eliminable_variables_tuple, rule="min_neighbors"),
                peak_factor_size=0,
            ),
            OrderingSummary(
                name="min_weight",
                ordering=_build_greedy_order(initial_scopes, eliminable_variables_tuple, rule="min_weight"),
                peak_factor_size=0,
            ),
            OrderingSummary(
                name="min_fill",
                ordering=_build_greedy_order(initial_scopes, eliminable_variables_tuple, rule="min_fill"),
                peak_factor_size=0,
            ),
        )
        heuristic_results = tuple(
            OrderingSummary(
                name=summary.name,
                ordering=summary.ordering,
                peak_factor_size=evaluate_ordering_peak_from_scopes(initial_scopes, summary.ordering),
            )
            for summary in heuristic_results
        )
    else:
        if not isinstance(heuristic_payload, list):
            raise ValueError("heuristic_results must be a list")
        heuristic_results = tuple(_ordering_summary_from_payload(item) for item in heuristic_payload)

    random_search_best_payload = payload.get("random_search_best")
    if random_search_best_payload is None:
        random_search_best = OrderingSummary(
            name="random_best_of_1000",
            ordering=baseline_ordering,
            peak_factor_size=baseline_peak_factor_size,
        )
    else:
        random_search_best = _ordering_summary_from_payload(random_search_best_payload)

    gold_ordering = payload.get("gold_ordering")
    if gold_ordering is None:
        gold_candidates = list(heuristic_results) + [random_search_best]
        gold = min(gold_candidates, key=lambda summary: (summary.peak_factor_size, summary.name, summary.ordering))
        gold_ordering_tuple = gold.ordering
        gold_peak_factor_size = gold.peak_factor_size
        gold_source = gold.name
    else:
        gold_ordering_tuple = _string_tuple(gold_ordering, "gold_ordering")
        gold_peak = payload.get("gold_peak_factor_size")
        if gold_peak is None:
            gold_peak_factor_size = evaluate_ordering_peak_from_scopes(initial_scopes, gold_ordering_tuple)
        else:
            gold_peak_factor_size = _optional_int(gold_peak, 0, "gold_peak_factor_size")
        gold_source = str(payload.get("gold_source", "provided"))

    exact_posterior_value = payload.get("exact_posterior")
    instance_stub = BayesianVEInstance(
        seed=seed,
        difficulty=difficulty,
        requested_total_variables=requested_total_variables,
        total_variables=total_variables,
        variables=variables,
        query_variable=query_variable,
        evidence=evidence,
        eliminable_variables=eliminable_variables_tuple,
        exact_posterior=0.0,
        baseline_ordering=baseline_ordering,
        baseline_peak_factor_size=baseline_peak_factor_size,
        gold_ordering=gold_ordering_tuple,
        gold_peak_factor_size=gold_peak_factor_size,
        gold_source=gold_source,
        heuristic_results=heuristic_results,
        random_search_best=random_search_best,
        min_fill_peak_factor_size=_optional_int(
            payload.get("min_fill_peak_factor_size"),
            _min_fill_peak(heuristic_results),
            "min_fill_peak_factor_size",
        ),
        problem_statement=str(payload.get("problem_statement", "")),
        hidden_clusters=_cluster_mapping(payload.get("hidden_clusters")),
        bridge_variables=_optional_string_tuple(payload.get("bridge_variables")),
        decoy_variable=str(payload.get("decoy_variable", "D0")),
        attempt_index=_optional_int(payload.get("attempt_index"), 0, "attempt_index"),
        size_escalated=bool(payload.get("size_escalated", total_variables != requested_total_variables)),
        scale_note=None if payload.get("scale_note") is None else str(payload.get("scale_note")),
    )

    if exact_posterior_value is None:
        exact_posterior, _ = evaluate_exact_probability(instance_stub, gold_ordering_tuple)
    else:
        exact_posterior = float(exact_posterior_value)

    problem_statement = instance_stub.problem_statement or render_problem(
        BayesianVEInstance(
            seed=instance_stub.seed,
            difficulty=instance_stub.difficulty,
            requested_total_variables=instance_stub.requested_total_variables,
            total_variables=instance_stub.total_variables,
            variables=instance_stub.variables,
            query_variable=instance_stub.query_variable,
            evidence=instance_stub.evidence,
            eliminable_variables=instance_stub.eliminable_variables,
            exact_posterior=exact_posterior,
            baseline_ordering=instance_stub.baseline_ordering,
            baseline_peak_factor_size=instance_stub.baseline_peak_factor_size,
            gold_ordering=instance_stub.gold_ordering,
            gold_peak_factor_size=instance_stub.gold_peak_factor_size,
            gold_source=instance_stub.gold_source,
            heuristic_results=instance_stub.heuristic_results,
            random_search_best=instance_stub.random_search_best,
            min_fill_peak_factor_size=instance_stub.min_fill_peak_factor_size,
            problem_statement="",
            hidden_clusters=instance_stub.hidden_clusters,
            bridge_variables=instance_stub.bridge_variables,
            decoy_variable=instance_stub.decoy_variable,
            attempt_index=instance_stub.attempt_index,
            size_escalated=instance_stub.size_escalated,
            scale_note=instance_stub.scale_note,
        )
    )

    return BayesianVEInstance(
        seed=instance_stub.seed,
        difficulty=instance_stub.difficulty,
        requested_total_variables=instance_stub.requested_total_variables,
        total_variables=instance_stub.total_variables,
        variables=instance_stub.variables,
        query_variable=instance_stub.query_variable,
        evidence=instance_stub.evidence,
        eliminable_variables=instance_stub.eliminable_variables,
        exact_posterior=exact_posterior,
        baseline_ordering=instance_stub.baseline_ordering,
        baseline_peak_factor_size=instance_stub.baseline_peak_factor_size,
        gold_ordering=instance_stub.gold_ordering,
        gold_peak_factor_size=instance_stub.gold_peak_factor_size,
        gold_source=instance_stub.gold_source,
        heuristic_results=instance_stub.heuristic_results,
        random_search_best=instance_stub.random_search_best,
        min_fill_peak_factor_size=instance_stub.min_fill_peak_factor_size,
        problem_statement=problem_statement,
        hidden_clusters=instance_stub.hidden_clusters,
        bridge_variables=instance_stub.bridge_variables,
        decoy_variable=instance_stub.decoy_variable,
        attempt_index=instance_stub.attempt_index,
        size_escalated=instance_stub.size_escalated,
        scale_note=instance_stub.scale_note,
    )

def _variable_from_payload(payload: dict[str, Any]) -> VariableSpec:
    if not isinstance(payload, dict):
        raise ValueError("each variable must be a dict")
    name = _required_str(payload.get("name"), "variable.name")
    parents = _string_tuple(payload.get("parents", []), f"{name}.parents")
    cpt_true_probs_raw = payload.get("cpt_true_probs")
    if not isinstance(cpt_true_probs_raw, list) or not cpt_true_probs_raw:
        raise ValueError(f"{name}.cpt_true_probs must be a non-empty list")
    cpt_true_probs = tuple(float(value) for value in cpt_true_probs_raw)
    return VariableSpec(name=name, parents=parents, cpt_true_probs=cpt_true_probs)

def _ordering_summary_from_payload(payload: dict[str, Any]) -> OrderingSummary:
    if not isinstance(payload, dict):
        raise ValueError("ordering summary must be a dict")
    return OrderingSummary(
        name=_required_str(payload.get("name"), "ordering_summary.name"),
        ordering=_string_tuple(payload.get("ordering"), "ordering_summary.ordering"),
        peak_factor_size=_optional_int(payload.get("peak_factor_size"), 0, "ordering_summary.peak_factor_size"),
    )

def _required_str(value: Any, field_name: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{field_name} must be a non-empty string")
    return value.strip()

def _string_tuple(value: Any, field_name: str) -> tuple[str, ...]:
    if not isinstance(value, list):
        raise ValueError(f"{field_name} must be a list")
    result = tuple(str(item).strip() for item in value)
    if not all(result):
        raise ValueError(f"{field_name} must not contain empty values")
    return result

def _optional_string_tuple(value: Any) -> tuple[str, ...]:
    if value is None:
        return ()
    if not isinstance(value, list):
        raise ValueError("expected a list of strings")
    return tuple(str(item).strip() for item in value if str(item).strip())

def _int_mapping(value: Any, field_name: str) -> dict[str, int]:
    if not isinstance(value, dict):
        raise ValueError(f"{field_name} must be a dict")
    parsed: dict[str, int] = {}
    for key, raw in value.items():
        parsed[str(key)] = int(raw)
    return parsed

def _cluster_mapping(value: Any) -> dict[str, tuple[str, ...]]:
    if value is None:
        return {}
    if not isinstance(value, dict):
        raise ValueError("hidden_clusters must be a dict")
    parsed: dict[str, tuple[str, ...]] = {}
    for key, raw in value.items():
        if not isinstance(raw, list):
            raise ValueError("hidden_clusters values must be lists")
        parsed[str(key)] = tuple(str(item) for item in raw)
    return parsed

def _optional_int(value: Any, default: int, field_name: str) -> int:
    if value is None:
        return default
    try:
        return int(value)
    except Exception as exc:
        raise ValueError(f"{field_name} must be an integer") from exc

def _min_fill_peak(heuristic_results: Sequence[OrderingSummary]) -> int:
    for summary in heuristic_results:
        if summary.name == "min_fill":
            return summary.peak_factor_size
    return 0

def _posterior_gap_pct(exact_posterior: float, submitted_probability: float) -> float:
    denominator = max(abs(exact_posterior), MIN_POSITIVE)
    return 100.0 * abs(submitted_probability - exact_posterior) / denominator

__all__ = ["build_instance_payload", "verify"]

# ── Verifier registry (built from inlined verify functions above) ─────────────
_SCHEMA_RE = _re.compile(r"BEST_GUESS(?: JSON)? schema:\s*(.*)", _re.IGNORECASE | _re.DOTALL)

CLASS_TO_VERIFIER: dict[str, Any] = {
    "cjs":      cjs_verify,
    "graphcol": graphcol_verify,
    "mwis":     mwis_verify,
    "steiner":  steiner_verify,
    "tsp":      tsp_verify,
    "ve":       ve_verify,
}

CLASS_TO_BEST_GUESS_SCHEMA: dict[str, str] = {}
for _cls, _fn in CLASS_TO_VERIFIER.items():
    _doc = inspect.getdoc(_fn) or ""
    _m = _SCHEMA_RE.search(_doc)
    if _m:
        _schema = _m.group(1).strip()
        if _schema:
            CLASS_TO_BEST_GUESS_SCHEMA[_cls] = _schema

# ──────────────────────────────────────────────────────────────────────
# harness/prompt.py
# ──────────────────────────────────────────────────────────────────────

import json
from typing import Any

CLASS_DISPLAY_NAMES = {
    "cjs": "Coupled Job-Shop",
    "graphcol": "Graph Coloring",
    "mwis": "Maximum Weighted Independent Set",
    "steiner": "Steiner x Coloring",
    "tsp": "Traveling Salesperson",
    "ve": "Bayesian Variable Elimination",
}

NO_MARKDOWN_WRAP_RULE = (
    "Do NOT wrap `PLAN_STATE`, `NEXT_SUB`, `UPDATED_PLAN_STATE`, `DECISION`, `BEST_GUESS`, "
    "`QUALITY_FORECAST`, or `CONTINUE_FORECAST` in markdown headers (`##`, `#`), bullets "
    "(`-`, `*`), or code fences (```). Emit each as a literal line starting with `<LABEL>:` "
    "followed by the value."
)

TURN1_ANTI_EXAMPLES = (
    "CORRECT: PLAN_STATE: Start from the baseline...\n"
    "WRONG:   ## PLAN_STATE\n"
    "Start from the baseline...  (markdown header — will not parse)\n"
    "WRONG:   - PLAN_STATE: Start from the baseline...   (bullet prefix — will not parse)"
)

EXEC_ANTI_EXAMPLES = (
    "CORRECT: UPDATED_PLAN_STATE: Keep a valid structured answer at every turn.\n"
    "WRONG:   ## UPDATED_PLAN_STATE\n"
    "Keep a valid structured answer at every turn.  (markdown header — will not parse)\n"
    'WRONG:   - BEST_GUESS: {"assignment": {"N01": 1, "N02": 2}}   (bullet prefix — will not parse)'
)

def build_system_prompt() -> str:
    return (
        "You are solving an optimization problem.\n\n"
        "Value. Each problem is worth up to 100 points of value. Your gap to the optimum "
        "reduces value linearly, at 1 point per percentage point of gap, floored at 0.\n\n"
        "Time cost. Wall time costs 0.01 points per second.\n\n"
        f"Budgets. Total wall budget is {TOTAL_BUDGET_S}s. Turn 1 planning budget is "
        f"{PLAN_TURN_BUDGET_S}s. Each execution subtask may budget up to {SUBTASK_BUDGET_S}s. "
        f"The harness reserves {CF_RESERVE_S}s for a forced one-more-turn counterfactual branch "
        "after a clean stop.\n\n"
        "Protocol. You may decompose the problem into subtasks, revise your plan each turn, "
        "and stop when you judge no more work is worth its time cost. The harness extracts "
        "structured fields from your raw text, so when a field expects JSON, emit valid JSON.\n\n"
        "Output budget. Your response is token-limited. Keep SUB_N sections under 30 lines — "
        "key moves and result only. You MUST reach all structured fields "
        "(BEST_GUESS, UPDATED_PLAN_STATE, QUALITY_FORECAST, CONTINUE_FORECAST, DECISION) "
        "in every exec turn.\n\n"
        "Do not call tools. Do not write code, pseudocode, or solver sketches. Work only from "
        "the prompt, the running transcript, the current best artifact, and your own reasoning."
    )

def build_turn1_prompt(instance_nl: str, *, cls: str) -> str:
    return (
        f"{instance_nl}\n\n"
        f"Turn 1 for {CLASS_DISPLAY_NAMES.get(cls, cls)} is planning only. Do not emit "
        "BEST_GUESS yet.\n\n"
        "Emit, in order:\n"
        "- `PLAN_STATE`: plan (≤ 15 lines). We will quote this back to you verbatim on every later turn.\n"
        "- `NEXT_SUB`: `{id: 1, desc, p_solve, time_budget_s}`. `p_solve` is your probability "
        "that the final answer lands within the target gap after this subtask. `time_budget_s` "
        f"must be <= {SUBTASK_BUDGET_S}.\n\n"
        "Example shape:\n"
        "PLAN_STATE: Start from the baseline, identify the main bottleneck, then spend one subtask "
        "trying to produce a cleaner feasible answer.\n"
        'NEXT_SUB: {"id": 1, "desc": "Construct one improved feasible candidate", '
        '"p_solve": 0.35, "time_budget_s": 180}\n\n'
        f"{NO_MARKDOWN_WRAP_RULE}\n"
        f"{TURN1_ANTI_EXAMPLES}\n\n"
        "Emit both labelled fields exactly once, in the listed order. Failing to emit either "
        "required field causes strict-parse failure, and no score is recorded for the turn. "
        "Do not emit BEST_GUESS yet.\n\n"
        f"Planning budget: {PLAN_TURN_BUDGET_S}s."
    )

def build_exec_prompt(
    *,
    instance_nl: str,
    cls: str,
    turn_index: int,
    transcript: str,
    elapsed_s: float,
    current_best: dict[str, Any] | None,
    last_subtask: dict[str, Any] | None,
    subtask_history: list[dict[str, Any]],
) -> str:
    remaining_s = max(0.0, TOTAL_BUDGET_S - elapsed_s)
    current_best_block = "CURRENT_BEST_JSON:\nnone yet"
    if current_best is not None:
        current_best_block = "CURRENT_BEST_JSON:\n" + json.dumps(current_best, indent=2, sort_keys=True)

    schema_block = _best_guess_schema_block(cls)
    timing_block = _timing_block(
        elapsed_s=elapsed_s,
        remaining_s=remaining_s,
        last_subtask=last_subtask,
        subtask_history=subtask_history,
    )

    return (
        f"{instance_nl}\n\n"
        "FULL PRIOR TRANSCRIPT:\n"
        f"{transcript.strip()}\n\n"
        f"{timing_block}\n\n"
        f"{current_best_block}\n\n"
        f"Exec turn {turn_index}. Emit, in order:\n"
        f"- `SUB_{turn_index - 1}`: key findings for this subtask (≤ 30 lines)\n"
        f"- `BEST_GUESS`: class-specific JSON for `{cls}`\n"
        f"{schema_block}\n"
        "- `UPDATED_PLAN_STATE`: revised plan, or keep the prior plan verbatim\n"
        "- `QUALITY_FORECAST`: `{p_gap_le_2, p_gap_le_5, p_gap_le_10}` for solo classes, "
        "or the VE-specific thresholds if this is `ve`\n"
        "- `CONTINUE_FORECAST`: `{p_improve, expected_delta_score}` and optionally "
        "`expected_gap_reduction`\n"
        "- `DECISION`: `continue` or `stop`\n"
        f"- `NEXT_SUB`: `{{id: {turn_index}, desc, p_solve, time_budget_s}}` only if you continue\n"
        "\n"
        "Example shape:\n"
        f"SUB_{turn_index - 1}: I checked one concrete local move and kept the best valid candidate found.\n"
        'BEST_GUESS: {"assignment": {"N01": 1, "N02": 2, "N03": 3}}\n'
        "UPDATED_PLAN_STATE: Keep a valid structured answer at every turn, then spend one more "
        "subtask on the highest-value local improvement.\n"
        'QUALITY_FORECAST: {"p_gap_le_2": 0.05, "p_gap_le_5": 0.2, "p_gap_le_10": 0.55}\n'
        'CONTINUE_FORECAST: {"p_improve": 0.35, "expected_delta_score": 1.8, "expected_gap_reduction": 4.0}\n'
        "DECISION: continue\n"
        f'NEXT_SUB: {{"id": {turn_index}, "desc": "Try one more improvement move", "p_solve": 0.3, "time_budget_s": 120}}\n\n'
        f"{NO_MARKDOWN_WRAP_RULE}\n"
        f"{EXEC_ANTI_EXAMPLES}\n\n"
        "All labelled fields must be emitted in the listed order. The `BEST_GUESS:` line must contain "
        "valid JSON, with no code fences and no extra text inside that field. Failing to emit any "
        "of `BEST_GUESS`, `UPDATED_PLAN_STATE`, `QUALITY_FORECAST`, `CONTINUE_FORECAST`, "
        "`DECISION`, or, when you continue, `NEXT_SUB` causes strict-parse failure, and no "
        "score is recorded for the turn. Keep each field concise enough that you still emit the "
        "full labelled output."
    )

def build_force_continue_prompt(
    *,
    instance_nl: str,
    cls: str,
    turn_index: int,
    transcript: str,
    elapsed_s: float,
    current_best: dict[str, Any] | None,
    last_subtask: dict[str, Any] | None,
    subtask_history: list[dict[str, Any]],
) -> str:
    base = build_exec_prompt(
        instance_nl=instance_nl,
        cls=cls,
        turn_index=turn_index,
        transcript=transcript,
        elapsed_s=elapsed_s,
        current_best=current_best,
        last_subtask=last_subtask,
        subtask_history=subtask_history,
    )
    return (
        f"{base}\n"
        "Counterfactual branch: take exactly one more execution turn from the same transcript state. "
        "Do not emit `DECISION` or `NEXT_SUB` on this branch."
    )

def _timing_block(
    *,
    elapsed_s: float,
    remaining_s: float,
    last_subtask: dict[str, Any] | None,
    subtask_history: list[dict[str, Any]],
) -> str:
    if last_subtask is None:
        last_summary = "none yet"
    else:
        budgeted_s = last_subtask.get("budgeted_s", "?")
        actual_s = last_subtask.get("actual_s", "?")
        desc = last_subtask.get("desc", "")
        last_summary = f"budgeted {budgeted_s}s, actual {actual_s}s, desc={desc}"

    history_bits = []
    for item in subtask_history:
        sub_id = item.get("id", "?")
        budgeted_s = item.get("budgeted_s", "?")
        actual_s = item.get("actual_s", "?")
        history_bits.append(f"s{sub_id} {actual_s}/{budgeted_s}s")
    history_text = ", ".join(history_bits) if history_bits else "none yet"

    return (
        f"TIMING: global elapsed {elapsed_s:.1f}s / {TOTAL_BUDGET_S}s "
        f"(remaining {remaining_s:.1f}s).\n"
        f"LAST_SUBTASK: {last_summary}\n"
        f"SUBTASK_HISTORY: [{history_text}]"
    )

def _best_guess_schema_block(cls: str) -> str:
    try:
        pass  # import inlined
    except Exception:
        return "- BEST_GUESS schema example unavailable until verifier registries are in place."

    schema = CLASS_TO_BEST_GUESS_SCHEMA.get(cls)
    if not schema:
        return f"- BEST_GUESS schema example unavailable for `{cls}`."
    return f"- BEST_GUESS schema example for `{cls}`:\n```json\n{schema}\n```"

# ──────────────────────────────────────────────────────────────────────
# harness/runner.py
# ──────────────────────────────────────────────────────────────────────

import json
import threading
import time
from contextlib import nullcontext
from typing import Any

try:
    from kaggle_benchmarks import chats as kbench_chats
    from kaggle_benchmarks import contexts as kbench_contexts
except Exception:  # pragma: no cover - local tests do not require kaggle_benchmarks.
    kbench_chats = None
    kbench_contexts = None

def run_instance(
    llm: Any,
    instance: dict[str, Any],
    cls: str,
    difficulty: str,
    seed: int,
    gold_objective: float,
    baseline_objective: float,
    value_cap: float,
    *,
    components: list[Any] | None = None,
) -> dict[str, Any]:
    del gold_objective, baseline_objective, value_cap  # encoded in the instance/components payloads.

    system_prompt = build_system_prompt()
    instance_nl = render_nl(instance, cls)
    run_start = time.monotonic()

    transcript: list[dict[str, str]] = []
    parsed: dict[str, Any] = {}
    cf_parsed: dict[str, Any] = {}
    error: str | None = None
    stop_reason = "error"
    n_exec_turns = 0

    current_best = _initial_best_guess(instance, cls=cls, components=components)
    current_eval = _evaluate_submission(
        cls=cls,
        instance=instance,
        submission=current_best,
        components=components,
    )
    last_emitted_best_guess = current_best
    last_eval = current_eval
    last_subtask: dict[str, Any] | None = None
    subtask_history: list[dict[str, Any]] = []

    with _chat_session(name=f"run-{cls}-{difficulty}-seed{seed}", system_prompt=system_prompt):
        try:
            plan_response = _call_model(
                llm=llm,
                prompt_text=build_turn1_prompt(instance_nl, cls=cls),
                timeout_s=PLAN_TURN_BUDGET_S,
            )
        except Exception as exc:  # noqa: BLE001
            plan_response = {"text": "", "timed_out": False, "wall_s": time.monotonic() - run_start}
            error = f"{type(exc).__name__}: {exc}"

        if plan_response["text"]:
            transcript.append({"role": "assistant", "content": plan_response["text"]})

        plan_parsed = None
        if error is None and not plan_response["timed_out"]:
            plan_parsed = parse_plan_turn(plan_response["text"], cls=cls)

        if error is not None:
            stop_reason = "error"
        elif plan_response["timed_out"]:
            stop_reason = "wall_budget"
            error = f"plan turn timed out after {PLAN_TURN_BUDGET_S}s"
        elif plan_parsed is None:
            stop_reason = "error"
            error = "plan turn parse failed"
        else:
            next_sub = plan_parsed.get("next_sub")
            if not next_sub:
                stop_reason = "error"
                error = "plan turn omitted NEXT_SUB"
            else:
                turn_index = 2
                while next_sub is not None:
                    if n_exec_turns >= MAX_EXEC_TURNS:
                        stop_reason = "turn_cap"
                        break

                    elapsed_s = time.monotonic() - run_start
                    expected_turn_s = int(next_sub.get("time_budget_s", 0))
                    if elapsed_s + expected_turn_s > TOTAL_BUDGET_S - CF_RESERVE_S:
                        stop_reason = "wall_budget"
                        break

                    exec_prompt = build_exec_prompt(
                        instance_nl=instance_nl,
                        cls=cls,
                        turn_index=turn_index,
                        transcript=_render_transcript(transcript),
                        elapsed_s=elapsed_s,
                        current_best=current_best,
                        last_subtask=last_subtask,
                        subtask_history=subtask_history,
                    )

                    try:
                        exec_response = _call_model(
                            llm=llm,
                            prompt_text=exec_prompt,
                            timeout_s=expected_turn_s,
                        )
                    except Exception as exc:  # noqa: BLE001
                        exec_response = {"text": "", "timed_out": False, "wall_s": time.monotonic() - run_start}
                        error = f"{type(exc).__name__}: {exc}"
                        stop_reason = "error"
                        break

                    if exec_response["text"]:
                        transcript.append({"role": "assistant", "content": exec_response["text"]})

                    n_exec_turns += 1
                    last_subtask = {
                        "id": next_sub["id"],
                        "desc": next_sub["desc"],
                        "budgeted_s": expected_turn_s,
                        "actual_s": exec_response["wall_s"],
                    }
                    subtask_history.append(last_subtask)

                    if exec_response["timed_out"]:
                        stop_reason = "wall_budget"
                        error = f"exec turn {turn_index} timed out after {expected_turn_s}s"
                        break

                    exec_parsed = parse_exec_turn(
                        exec_response["text"],
                        cls=cls,
                        expected_subtask_id=next_sub["id"],
                    )
                    if exec_parsed is None:
                        stop_reason = "error"
                        error = f"exec turn {turn_index} parse failed"
                        break

                    parsed = exec_parsed
                    last_emitted_best_guess = exec_parsed["best_guess"]
                    last_eval = _evaluate_submission(
                        cls=cls,
                        instance=instance,
                        submission=last_emitted_best_guess,
                        components=components,
                    )

                    if _is_prompt_worthy(last_eval):
                        current_best = last_eval["submission"]
                        current_eval = last_eval

                    if exec_parsed["decision"] == "stop":
                        stop_reason = "decision_stop"
                        break

                    next_sub = exec_parsed["next_sub"]
                    turn_index += 1

    stop_eval = last_eval if last_eval is not None else current_eval
    score_at_stop = _score_evaluation(stop_eval, wall_s=time.monotonic() - run_start)
    score_after_cf = score_at_stop

    if stop_reason == "decision_stop":
        cf_turn_index = n_exec_turns + 2
        cf_timeout_s = min(CF_RESERVE_S, max(0, int(TOTAL_BUDGET_S - (time.monotonic() - run_start))))
        if cf_timeout_s >= 1:
            cf_prompt = build_force_continue_prompt(
                instance_nl=instance_nl,
                cls=cls,
                turn_index=cf_turn_index,
                transcript=_render_transcript(transcript),
                elapsed_s=time.monotonic() - run_start,
                current_best=last_emitted_best_guess,
                last_subtask=last_subtask,
                subtask_history=subtask_history,
            )
            try:
                cf_response = _call_model(llm=llm, prompt_text=cf_prompt, timeout_s=cf_timeout_s)
            except Exception as exc:  # noqa: BLE001
                cf_response = {"text": "", "timed_out": False, "wall_s": time.monotonic() - run_start}
                if error is None:
                    error = f"{type(exc).__name__}: {exc}"

            if cf_response["text"]:
                transcript.append({"role": "assistant", "content": cf_response["text"]})

            if cf_response["timed_out"]:
                error = error or f"counterfactual turn timed out after {cf_timeout_s}s"
            else:
                parsed_cf = parse_exec_turn(
                    cf_response["text"],
                    cls=cls,
                    expected_subtask_id=cf_turn_index - 1,
                    require_decision=False,
                )
                if parsed_cf is None:
                    error = error or "counterfactual turn parse failed"
                else:
                    cf_parsed = parsed_cf
                    cf_eval = _evaluate_submission(
                        cls=cls,
                        instance=instance,
                        submission=parsed_cf["best_guess"],
                        components=components,
                    )
                    score_after_cf = _score_evaluation(cf_eval, wall_s=time.monotonic() - run_start)
        else:
            error = error or "counterfactual turn skipped because no wall budget remained"

    wall_s = time.monotonic() - run_start
    cf_delta = score_after_cf - score_at_stop
    return {
        "score": score_after_cf,
        "score_at_stop": score_at_stop,
        "score_after_cf": score_after_cf,
        "cf_delta": cf_delta,
        "wall_s": wall_s,
        "n_exec_turns": n_exec_turns,
        "stop_reason": stop_reason,
        "transcript": transcript,
        "parsed": parsed,
        "cf_parsed": cf_parsed,
        "error": error,
    }

def _chat_session(*, name: str, system_prompt: str):
    if kbench_chats is None:
        return nullcontext()
    return kbench_chats.new(name=name, system_instructions=system_prompt)

def _call_model(llm: Any, prompt_text: str, timeout_s: int) -> dict[str, Any]:
    start = time.monotonic()
    result: list[Any] = [None]
    error: list[BaseException | None] = [None]
    current_context = kbench_contexts.get_current() if kbench_contexts is not None else None

    def _invoke() -> None:
        try:
            if current_context is not None and kbench_contexts is not None:
                kbench_contexts.set_from_context(current_context)
            result[0] = _prompt_once(llm, prompt_text)
        except BaseException as exc:  # noqa: BLE001
            error[0] = exc

    thread = threading.Thread(target=_invoke, daemon=True)
    thread.start()
    thread.join(timeout=timeout_s)
    wall_s = time.monotonic() - start

    if thread.is_alive():
        return {"text": "", "timed_out": True, "wall_s": wall_s}
    if error[0] is not None:
        raise error[0]

    raw = result[0]
    if isinstance(raw, str):
        text = raw
    else:
        text = str(getattr(raw, "content", raw) or "")
    return {"text": text.strip(), "timed_out": wall_s > timeout_s, "wall_s": wall_s}

def _prompt_once(llm: Any, prompt_text: str) -> Any:
    if hasattr(llm, "prompt"):
        prompt = getattr(llm, "prompt")
        try:
            return prompt(prompt_text, temperature=0)
        except TypeError:
            return prompt(prompt_text)
    if callable(llm):
        return llm(prompt_text)
    raise TypeError("llm must expose .prompt(message) or be callable")

def _render_transcript(transcript: list[dict[str, str]]) -> str:
    chunks = []
    for index, entry in enumerate(transcript, start=1):
        chunks.append(f"TURN_{index}_MODEL_RESPONSE:\n{entry['content']}")
    return "\n\n".join(chunks)

def _initial_best_guess(
    instance: dict[str, Any],
    *,
    cls: str,
    components: list[Any] | None,
) -> dict[str, Any] | None:
    baseline = instance.get("baseline_submission")
    if isinstance(baseline, dict):
        return baseline
    if cls != "portfolio" or not components:
        return None

    answers: dict[str, Any] = {}
    for component in components:
        if not isinstance(component, dict):
            continue
        problem_id = component.get("problem_id")
        sub_instance = component.get("sub_instance")
        if not isinstance(problem_id, str) or not isinstance(sub_instance, dict):
            continue
        baseline_submission = sub_instance.get("baseline_submission")
        if baseline_submission is not None:
            answers[problem_id] = baseline_submission
    return answers or None

def _evaluate_submission(
    *,
    cls: str,
    instance: dict[str, Any],
    submission: Any,
    components: list[Any] | None,
) -> dict[str, Any]:
    if cls == "portfolio":
        return _evaluate_portfolio_submission(submission=submission, components=components)

    verifier = CLASS_TO_VERIFIER.get(cls)
    if verifier is None:
        return {
            "mode": "solo",
            "gap_pct": 100.0,
            "feasible": False,
            "details": {"failure_reason": f"missing verifier for class {cls}"},
            "submission": submission if isinstance(submission, dict) else None,
        }

    gap_pct, feasible, details = verifier(instance, submission)
    normalized_submission = details.get("normalized_submission", submission)
    return {
        "mode": "solo",
        "gap_pct": float(gap_pct),
        "feasible": bool(feasible),
        "details": details,
        "submission": normalized_submission if isinstance(normalized_submission, dict) else submission,
    }

def _evaluate_portfolio_submission(
    *,
    submission: Any,
    components: list[Any] | None,
) -> dict[str, Any]:
    if not isinstance(submission, dict):
        return {
            "mode": "portfolio",
            "submission": {},
            "components": [],
            "details": {"failure_reason": "portfolio submission must be an object"},
        }

    raw_answers = submission.get("answers", submission)
    if not isinstance(raw_answers, dict):
        raw_answers = {}

    component_rows: list[dict[str, Any]] = []
    normalized_submission: dict[str, Any] = {}
    for component in components or []:
        if not isinstance(component, dict):
            continue

        problem_id = str(component.get("problem_id", "")).strip()
        component_cls = component.get("class")
        sub_instance = component.get("sub_instance")
        value_cap = float(component.get("value_cap", 0.0))
        verifier = CLASS_TO_VERIFIER.get(component_cls) if isinstance(component_cls, str) else None
        raw_component_submission = raw_answers.get(problem_id)

        if verifier is None or not isinstance(sub_instance, dict):
            component_rows.append(
                {
                    "problem_id": problem_id,
                    "class": component_cls,
                    "gap_pct": 100.0,
                    "feasible": False,
                    "value_cap": value_cap,
                    "details": {"failure_reason": f"missing verifier or instance for {problem_id}"},
                }
            )
            continue

        gap_pct, feasible, details = verifier(sub_instance, raw_component_submission)
        normalized_component = details.get("normalized_submission", raw_component_submission)
        normalized_submission[problem_id] = normalized_component
        component_rows.append(
            {
                "problem_id": problem_id,
                "class": component_cls,
                "gap_pct": float(gap_pct),
                "feasible": bool(feasible),
                "value_cap": value_cap,
                "details": details,
            }
        )

    return {
        "mode": "portfolio",
        "submission": normalized_submission,
        "components": component_rows,
        "details": {"components": component_rows},
    }

def _is_prompt_worthy(evaluation: dict[str, Any]) -> bool:
    if evaluation["mode"] == "portfolio":
        return bool(evaluation["components"])
    return bool(evaluation["feasible"]) and isinstance(evaluation["submission"], dict)

def _score_evaluation(evaluation: dict[str, Any], *, wall_s: float) -> float:
    if evaluation["mode"] == "portfolio":
        return score_portfolio(evaluation["components"], wall_s)
    return score_solo(
        gap_pct=float(evaluation["gap_pct"]),
        feasible=bool(evaluation["feasible"]),
        wall_s=wall_s,
    )


def _is_missing(value: Any) -> bool:
    if value is None or type(value).__name__ == "NAType":
        return True
    try:
        return bool(value != value)
    except Exception:
        return False


def _coerce_json(value: Any, *, default: Any = None) -> Any:
    if _is_missing(value):
        return default
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return default
        return _json.loads(stripped)
    return value


def _normalize_components(value: Any) -> list[dict[str, Any]] | None:
    parsed = _coerce_json(value, default=None)
    if parsed is None:
        return None
    if not isinstance(parsed, list):
        raise TypeError("components must be a list, JSON string, or null")
    return parsed


@kbench.task(name="meta_hch_bench")
def run(
    llm: Any,
    id: str,
    class_: str,
    difficulty: str,
    seed: int,
    instance: Any,
    gold_objective: float,
    baseline_objective: float,
    value_cap: float,
    wall_budget_s: int = 1800,
    components: Any = None,
) -> float:
    del id, wall_budget_s

    inst = _coerce_json(instance, default=None)
    if not isinstance(inst, dict):
        raise TypeError("instance must be a dict or JSON object string")

    row = run_instance(
        llm=llm,
        instance=inst,
        cls=str(class_),
        difficulty=str(difficulty),
        seed=int(seed),
        gold_objective=float(gold_objective),
        baseline_objective=float(baseline_objective),
        value_cap=float(value_cap),
        components=_normalize_components(components),
    )
    return float(row["score"])


In [ ]:
# Explicitly invoke the registered task so kbench writes a .run.json alongside
# the .task.json. The @kbench.task decorator only REGISTERS the task; execution
# (which produces the run artifact) requires Task.run() / Task.evaluate().
import inspect
import kaggle_benchmarks as kbench
from kaggle_benchmarks import kaggle as _kbk

if not _kbk.is_configured():
    print("[meta-hch-bench] Kaggle model proxy is not configured (MODEL_PROXY_URL / "
          "MODEL_PROXY_API_KEY missing). Skipping run. The @kbench.task decorator has "
          "already registered the task; a run will be triggered by the hackathon harness.")
else:
    _tasks = [t for t in kbench.client.registry.values()]
    if len(_tasks) != 1:
        raise RuntimeError(f"Expected exactly one registered @kbench.task, found {len(_tasks)}: "
                           f"{[t.name for t in _tasks]}")
    _t = _tasks[0]
    _nparams = len(inspect.signature(_t.func).parameters)
    _run = _t.run(kbench.llm) if _nparams >= 1 else _t.run()
    print(f"task={_t.name} status={getattr(_run.status, 'name', _run.status)} "
          f"passed={_run.passed} result={_run.result!r}")